# 09 — RQ1 transformer experiments (tuning)

Documents how the frozen RQ1 configurations in notebook 10 were chosen — the learning-rate search (1e-5 / 2e-5 / 3e-5), the regularisation comparison (baseline vs weight decay vs weight decay + warmup), step-level checkpoint selection, validation **threshold tuning**, and confirmation of the winning setup across seeds 42/1/2, for RoBERTa-base and DeBERTa-v3-base. The hierarchical (nested-1B) experiments and the summary figures follow. All tuning is on `train_fit` (fit) and `validation` (selection) only — no test access.

**Requires a GPU.** This notebook is a record of the search; it is slow to re-run end-to-end. Paths are repo-relative (`data/processed/`, model runs under `final_training_runs/`).

In [ ]:
# Installs

!pip install -q sentencepiece protobuf



## Fine-tuning RoBERTa-base

RoBERTa-base is tuned for all three constructs with a fixed procedure: (1) a learning-rate search over 1e-5 / 2e-5 / 3e-5; (2) a regularisation comparison (no decay vs weight decay 0.01 vs weight decay + 10% warmup); (3) step-level checkpoint selection on validation; (4) decision-threshold tuning on validation; and (5) confirmation of the winning configuration across seeds 42, 1 and 2. Fitting uses `train_fit`; every selection decision uses `validation`; the test set is never touched here. RoBERTa-base was selected as the frozen model for **construct 1A**.

In [ ]:
# ============================================================
# RoBERTa-base — hyperparameter tuning (RQ1)
#
# Tunes RoBERTa-base for each construct (1A, 1B, 2) on train_fit,
# selecting every choice on validation. test.csv is never used.
#
# Procedure:
#   1. Learning-rate search: 1e-5 / 2e-5 / 3e-5
#   2. Regularisation comparison at the best LR:
#        baseline (no decay) / weight decay 0.01 / decay + 10% warmup
#   3. Step-level checkpoint selection (best validation F1)
#   4. Decision-threshold tuning on validation
#   5. Confirm the winning setup across seeds 42, 1, 2
#
# Inputs: data/processed/train_fit.csv, validation.csv
# ============================================================

import gc
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_score, recall_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)


# ============================================================
# 1. FILES
# ============================================================

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "final_training_runs" / "models_roberta"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_fit.csv"
VAL_PATH   = DATA_DIR / "validation.csv"

if not TRAIN_PATH.exists():
    raise FileNotFoundError("data/processed/train_fit.csv not found")

if not VAL_PATH.exists():
    raise FileNotFoundError("data/processed/validation.csv not found")

train_fit = pd.read_csv(TRAIN_PATH)
val       = pd.read_csv(VAL_PATH)

LABELS = ["label_1A", "label_1B", "label_2"]

print("train_fit rows:", len(train_fit))
print("validation rows:", len(val))

for label in LABELS:
    print(
        f"{label}: "
        f"train positives={int(train_fit[label].sum())}, "
        f"validation positives={int(val[label].sum())}"
    )


# ============================================================
# 2. SETTINGS
# ============================================================

MODEL_NAME = "roberta-base"

LEARNING_RATES = [1e-5, 2e-5, 3e-5]

SEEDS = [42, 1, 2]

THRESHOLDS = np.arange(0.10, 0.91, 0.01)

REGULARISATION = {
    "baseline": {
        "weight_decay": 0.00,
        "warmup_ratio": 0.00
    },

    "weight_decay": {
        "weight_decay": 0.01,
        "warmup_ratio": 0.00
    },

    "weight_decay_warmup": {
        "weight_decay": 0.01,
        "warmup_ratio": 0.10
    }
}


# T4 should handle RoBERTa with batch 16.
TRAIN_BATCH = 16
EVAL_BATCH = 32
GRAD_ACCUM = 1

print("CUDA available:", torch.cuda.is_available())
print("Effective training batch:", TRAIN_BATCH * GRAD_ACCUM)


# ============================================================
# 3. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ============================================================
# 4. METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "precision": precision_score(
            labels, preds, zero_division=0
        ),
        "recall": recall_score(
            labels, preds, zero_division=0
        ),
        "f1": f1_score(
            labels, preds, zero_division=0
        )
    }


# ============================================================
# 5. DATASETS
# ============================================================

def make_ds(df, label):

    ds = Dataset.from_pandas(
        df[["text_clean", label]]
        .rename(columns={label: "labels"})
    )

    ds = ds.map(
        lambda batch: tokenizer(
            batch["text_clean"],
            truncation=True,
            max_length=256
        ),
        batched=True
    )

    return ds


DATASETS = {}
CLASS_WEIGHTS = {}

for label in LABELS:

    DATASETS[label] = (
        make_ds(train_fit, label),
        make_ds(val, label)
    )

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=train_fit[label].values
    )

    CLASS_WEIGHTS[label] = torch.tensor(
        weights,
        dtype=torch.float32
    )


# ============================================================
# 6. THRESHOLD SEARCH
# ============================================================

def tune_threshold(y_true, probabilities):

    scores = []

    for threshold in THRESHOLDS:

        preds = (
            probabilities >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            preds,
            zero_division=0
        )

        scores.append(score)

    best_idx = int(np.argmax(scores))

    return (
        float(THRESHOLDS[best_idx]),
        float(scores[best_idx])
    )


# ============================================================
# 7. MEMORY CLEANUP
# ============================================================

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 8. TRAIN ONE MODEL
# ============================================================

def train_one(label, seed, learning_rate, config_name):

    clear_memory()
    set_seed(seed)

    train_ds, val_ds = DATASETS[label]
    class_weights = CLASS_WEIGHTS[label]

    reg = REGULARISATION[config_name]


    class WeightedTrainer(Trainer):

        def compute_loss(
            self,
            model,
            inputs,
            return_outputs=False,
            **kwargs
        ):

            labels = inputs.pop("labels")
            outputs = model(**inputs)

            # Custom weighted CE in float32
            logits = outputs.logits.float()

            weights_device = class_weights.to(
                device=logits.device,
                dtype=torch.float32
            )

            loss = nn.CrossEntropyLoss(
                weight=weights_device
            )(
                logits,
                labels
            )

            return (
                (loss, outputs)
                if return_outputs
                else loss
            )


    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )


    with tempfile.TemporaryDirectory(
        dir=MODEL_DIR,
        prefix=f"tmp_{label}_"
    ) as output_dir:


        args = TrainingArguments(

            output_dir=output_dir,

            num_train_epochs=3,

            per_device_train_batch_size=TRAIN_BATCH,
            per_device_eval_batch_size=EVAL_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM,

            learning_rate=learning_rate,

            weight_decay=reg["weight_decay"],
            warmup_ratio=reg["warmup_ratio"],

            eval_strategy="steps",
            save_strategy="steps",

            eval_steps=25,
            save_steps=25,

            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,

            save_total_limit=1,

            logging_steps=25,
            report_to="none",

            seed=seed,
            data_seed=seed
        )


        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics
        )


        trainer.train()


        # ------------------------------------
        # Predictions from best checkpoint
        # ------------------------------------

        output = trainer.predict(val_ds)

        checkpoint_f1 = float(
            trainer.state.best_metric
        )

        y_true = output.label_ids

        probabilities = torch.softmax(
            torch.tensor(output.predictions),
            dim=1
        ).numpy()[:, 1]


        # ------------------------------------
        # Default 0.50 threshold
        # ------------------------------------

        default_preds = (
            probabilities >= 0.50
        ).astype(int)

        default_precision = precision_score(
            y_true,
            default_preds,
            zero_division=0
        )

        default_recall = recall_score(
            y_true,
            default_preds,
            zero_division=0
        )

        default_f1 = f1_score(
            y_true,
            default_preds,
            zero_division=0
        )


        # ------------------------------------
        # Tuned threshold
        # ------------------------------------

        best_threshold, tuned_f1 = tune_threshold(
            y_true,
            probabilities
        )

        tuned_preds = (
            probabilities >= best_threshold
        ).astype(int)

        tuned_precision = precision_score(
            y_true,
            tuned_preds,
            zero_division=0
        )

        tuned_recall = recall_score(
            y_true,
            tuned_preds,
            zero_division=0
        )


    result = {
        "label": label,
        "seed": seed,
        "learning_rate": learning_rate,
        "config": config_name,

        "weight_decay":
            reg["weight_decay"],

        "warmup_ratio":
            reg["warmup_ratio"],

        "checkpoint_f1":
            checkpoint_f1,

        "default_precision":
            float(default_precision),

        "default_recall":
            float(default_recall),

        "default_f1":
            float(default_f1),

        "best_threshold":
            float(best_threshold),

        "tuned_precision":
            float(tuned_precision),

        "tuned_recall":
            float(tuned_recall),

        "tuned_f1":
            float(tuned_f1)
    }


    del trainer
    del model
    del output

    clear_memory()

    return result


# ============================================================
# STAGE A — LEARNING RATE SEARCH
# Seed 42 only
# ============================================================

print("\n\n========================================")
print("STAGE A — ROBERTA LEARNING RATE SEARCH")
print("========================================")


baseline_winner = {}
best_lr = {}

lr_search_results = []


for label in LABELS:

    print(f"\n========== {label} ==========")

    candidates = []


    for lr in LEARNING_RATES:

        print(
            f"\n{label} | LR={lr:.0e} | baseline"
        )

        result = train_one(
            label=label,
            seed=42,
            learning_rate=lr,
            config_name="baseline"
        )

        candidates.append(result)
        lr_search_results.append(result)

        print(
            f"Default F1 = {result['default_f1']:.3f}"
        )

        print(
            f"Threshold  = {result['best_threshold']:.2f}"
        )

        print(
            f"Tuned F1   = {result['tuned_f1']:.3f}"
        )


    winner = max(
        candidates,
        key=lambda x: x["tuned_f1"]
    )

    baseline_winner[label] = winner

    best_lr[label] = winner[
        "learning_rate"
    ]

    print(
        f"\n>>> BEST LR {label}: "
        f"{best_lr[label]:.0e}"
    )


# ============================================================
# STAGE B — REGULARISATION SEARCH
# Seed 42, winning LR
# ============================================================

print("\n\n========================================")
print("STAGE B — ROBERTA REGULARISATION SEARCH")
print("========================================")


winning_seed42 = {}
best_config = {}


for label in LABELS:

    print(f"\n========== {label} ==========")

    # Include the already-run baseline winner
    candidates = [
        baseline_winner[label]
    ]


    for config_name in [
        "weight_decay",
        "weight_decay_warmup"
    ]:

        print(
            f"\n{label}"
            f" | LR={best_lr[label]:.0e}"
            f" | {config_name}"
        )

        result = train_one(
            label=label,
            seed=42,
            learning_rate=best_lr[label],
            config_name=config_name
        )

        candidates.append(result)

        print(
            f"Default F1 = {result['default_f1']:.3f}"
        )

        print(
            f"Threshold  = {result['best_threshold']:.2f}"
        )

        print(
            f"Tuned F1   = {result['tuned_f1']:.3f}"
        )


    # Baseline occurs first, so exact ties favour simpler config
    winner = max(
        candidates,
        key=lambda x: x["tuned_f1"]
    )

    winning_seed42[label] = winner
    best_config[label] = winner["config"]

    print(
        f"\n>>> WINNER {label}: "
        f"{best_config[label]}"
        f" | LR={best_lr[label]:.0e}"
        f" | tuned F1={winner['tuned_f1']:.3f}"
    )


# ============================================================
# STAGE C — 3-SEED CONFIRMATION
# ============================================================

print("\n\n========================================")
print("STAGE C — ROBERTA 3-SEED CONFIRMATION")
print("========================================")


seed_results = []


for label in LABELS:

    print(f"\n========== {label} ==========")

    # Reuse seed 42 result
    seed_results.append(
        winning_seed42[label]
    )


    for seed in [1, 2]:

        print(
            f"\n{label}"
            f" | seed={seed}"
            f" | LR={best_lr[label]:.0e}"
            f" | {best_config[label]}"
        )

        result = train_one(
            label=label,
            seed=seed,
            learning_rate=best_lr[label],
            config_name=best_config[label]
        )

        seed_results.append(result)

        print(
            f"Tuned F1 = {result['tuned_f1']:.3f}"
        )


# ============================================================
# STAGE D — FINAL SUMMARY
# ============================================================

results_df = pd.DataFrame(seed_results)

summary_rows = []


print("\n\n========================================")
print("FINAL ROBERTA VALIDATION SUMMARY")
print("========================================")


for label in LABELS:

    subset = results_df[
        results_df["label"] == label
    ]

    summary = {
        "label":
            label,

        "best_learning_rate":
            best_lr[label],

        "best_config":
            best_config[label],

        "weight_decay":
            subset["weight_decay"].iloc[0],

        "warmup_ratio":
            subset["warmup_ratio"].iloc[0],

        "default_f1_mean":
            subset["default_f1"].mean(),

        "default_f1_sd":
            subset["default_f1"].std(ddof=0),

        "tuned_f1_mean":
            subset["tuned_f1"].mean(),

        "tuned_f1_sd":
            subset["tuned_f1"].std(ddof=0),

        "median_threshold":
            subset["best_threshold"].median()
    }

    summary_rows.append(summary)


    print(f"\n{label}")

    print(
        f"  Best LR: "
        f"{summary['best_learning_rate']:.0e}"
    )

    print(
        f"  Config: "
        f"{summary['best_config']}"
    )

    print(
        f"  Weight decay: "
        f"{summary['weight_decay']}"
    )

    print(
        f"  Warmup ratio: "
        f"{summary['warmup_ratio']}"
    )

    print(
        f"  Default F1: "
        f"{summary['default_f1_mean']:.3f}"
        f" ± {summary['default_f1_sd']:.3f}"
    )

    print(
        f"  Tuned F1: "
        f"{summary['tuned_f1_mean']:.3f}"
        f" ± {summary['tuned_f1_sd']:.3f}"
    )

    print(
        f"  Thresholds: "
        f"{[round(x, 2) for x in subset['best_threshold']]}"
    )

    print(
        f"  Median threshold: "
        f"{summary['median_threshold']:.2f}"
    )


summary_df = pd.DataFrame(summary_rows)


# ============================================================
# SAVE CSV RESULTS TO data/processed/
# ============================================================

results_path = (
    DATA_DIR
    / "results_roberta_finetuned_validation.csv"
)

summary_path = (
    DATA_DIR
    / "results_roberta_finetuned_summary.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)


print("\nSaved:")
print(results_path)
print(summary_path)


# ============================================================
# DETAILED RESULTS TABLE
# ============================================================

display_df = results_df.copy()

display_df["learning_rate"] = (
    display_df["learning_rate"]
    .map(lambda x: f"{x:.0e}")
)


print("\n\n========================================")
print("DETAILED ROBERTA 3-SEED RESULTS")
print("========================================")


print(
    display_df[
        [
            "label",
            "seed",
            "learning_rate",
            "config",
            "weight_decay",
            "warmup_ratio",
            "checkpoint_f1",
            "default_precision",
            "default_recall",
            "default_f1",
            "best_threshold",
            "tuned_precision",
            "tuned_recall",
            "tuned_f1"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

## Fine-tuning DeBERTa-v3-base

DeBERTa-v3-base is tuned with the identical procedure on `train_fit` / `validation` only. It was selected as the frozen model for **constructs 1B and 2**, where its contextual representation outperformed RoBERTa.

In [ ]:
# ============================================================
# DeBERTa-v3-base — hyperparameter tuning (RQ1)
#
# Identical procedure to the RoBERTa cell above (learning-rate
# search, regularisation comparison, checkpoint selection,
# threshold tuning, multi-seed confirmation), on train_fit /
# validation only. test.csv is never used.
#
# Inputs: data/processed/train_fit.csv, validation.csv
# ============================================================

import gc
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_score, recall_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)


# ============================================================
# 1. FILES
# ============================================================

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "final_training_runs" / "models_deberta"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_fit.csv"
VAL_PATH   = DATA_DIR / "validation.csv"

if not TRAIN_PATH.exists():
    raise FileNotFoundError("data/processed/train_fit.csv not found")

if not VAL_PATH.exists():
    raise FileNotFoundError("data/processed/validation.csv not found")

train_fit = pd.read_csv(TRAIN_PATH)
val       = pd.read_csv(VAL_PATH)

LABELS = ["label_1A", "label_1B", "label_2"]

print("train_fit rows:", len(train_fit))
print("validation rows:", len(val))

for label in LABELS:
    print(
        f"{label}: "
        f"train positives={int(train_fit[label].sum())}, "
        f"validation positives={int(val[label].sum())}"
    )


# ============================================================
# 2. SETTINGS
# ============================================================

MODEL_NAME = "microsoft/deberta-v3-base"

LEARNING_RATES = [1e-5, 2e-5, 3e-5]

SEEDS = [42, 1, 2]

THRESHOLDS = np.arange(0.10, 0.91, 0.01)

REGULARISATION = {
    "baseline": {
        "weight_decay": 0.00,
        "warmup_ratio": 0.00
    },

    "weight_decay": {
        "weight_decay": 0.01,
        "warmup_ratio": 0.00
    },

    "weight_decay_warmup": {
        "weight_decay": 0.01,
        "warmup_ratio": 0.10
    }
}


# Smaller physical batch for DeBERTa.
# Effective training batch remains 16.
TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 2

print("CUDA available:", torch.cuda.is_available())
print("Effective training batch:", TRAIN_BATCH * GRAD_ACCUM)


# ============================================================
# 3. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ============================================================
# 4. METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "precision": precision_score(
            labels,
            preds,
            zero_division=0
        ),

        "recall": recall_score(
            labels,
            preds,
            zero_division=0
        ),

        "f1": f1_score(
            labels,
            preds,
            zero_division=0
        )
    }


# ============================================================
# 5. DATASETS
# ============================================================

def make_ds(df, label):

    ds = Dataset.from_pandas(
        df[["text_clean", label]]
        .rename(columns={label: "labels"})
    )

    ds = ds.map(
        lambda batch: tokenizer(
            batch["text_clean"],
            truncation=True,
            max_length=256
        ),
        batched=True
    )

    return ds


DATASETS = {}
CLASS_WEIGHTS = {}


for label in LABELS:

    DATASETS[label] = (
        make_ds(train_fit, label),
        make_ds(val, label)
    )

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=train_fit[label].values
    )

    CLASS_WEIGHTS[label] = torch.tensor(
        weights,
        dtype=torch.float32
    )


# ============================================================
# 6. THRESHOLD SEARCH
# ============================================================

def tune_threshold(y_true, probabilities):

    scores = []

    for threshold in THRESHOLDS:

        preds = (
            probabilities >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            preds,
            zero_division=0
        )

        scores.append(score)


    best_idx = int(
        np.argmax(scores)
    )

    return (
        float(THRESHOLDS[best_idx]),
        float(scores[best_idx])
    )


# ============================================================
# 7. MEMORY CLEANUP
# ============================================================

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 8. TRAIN ONE MODEL
# ============================================================

def train_one(label, seed, learning_rate, config_name):

    clear_memory()
    set_seed(seed)

    train_ds, val_ds = DATASETS[label]
    class_weights = CLASS_WEIGHTS[label]

    reg = REGULARISATION[
        config_name
    ]


    class WeightedTrainer(Trainer):

        def compute_loss(
            self,
            model,
            inputs,
            return_outputs=False,
            **kwargs
        ):

            labels = inputs.pop("labels")
            outputs = model(**inputs)

            # Explicit FP32 custom loss
            logits = outputs.logits.float()

            weights_device = class_weights.to(
                device=logits.device,
                dtype=torch.float32
            )

            loss = nn.CrossEntropyLoss(
                weight=weights_device
            )(
                logits,
                labels
            )

            return (
                (loss, outputs)
                if return_outputs
                else loss
            )


    # Explicit FP32 load because earlier DeBERTa
    # mixed-precision run produced NaNs.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        dtype=torch.float32
    )


    with tempfile.TemporaryDirectory(
        dir=MODEL_DIR,
        prefix=f"tmp_{label}_"
    ) as output_dir:


        args = TrainingArguments(

            output_dir=output_dir,

            num_train_epochs=3,

            per_device_train_batch_size=
                TRAIN_BATCH,

            per_device_eval_batch_size=
                EVAL_BATCH,

            gradient_accumulation_steps=
                GRAD_ACCUM,

            learning_rate=
                learning_rate,

            weight_decay=
                reg["weight_decay"],

            warmup_ratio=
                reg["warmup_ratio"],

            eval_strategy="steps",
            save_strategy="steps",

            eval_steps=25,
            save_steps=25,

            load_best_model_at_end=True,

            metric_for_best_model="f1",
            greater_is_better=True,

            save_total_limit=1,

            logging_steps=25,

            report_to="none",

            seed=seed,
            data_seed=seed,

            fp16=False,
            bf16=False
        )


        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics
        )


        trainer.train()


        # ------------------------------------
        # Validation prediction
        # ------------------------------------

        output = trainer.predict(
            val_ds
        )

        checkpoint_f1 = float(
            trainer.state.best_metric
        )

        y_true = output.label_ids

        probabilities = torch.softmax(
            torch.tensor(
                output.predictions
            ),
            dim=1
        ).numpy()[:, 1]


        # ------------------------------------
        # Default 0.50 threshold
        # ------------------------------------

        default_preds = (
            probabilities >= 0.50
        ).astype(int)

        default_precision = precision_score(
            y_true,
            default_preds,
            zero_division=0
        )

        default_recall = recall_score(
            y_true,
            default_preds,
            zero_division=0
        )

        default_f1 = f1_score(
            y_true,
            default_preds,
            zero_division=0
        )


        # ------------------------------------
        # Tuned threshold
        # ------------------------------------

        best_threshold, tuned_f1 = tune_threshold(
            y_true,
            probabilities
        )

        tuned_preds = (
            probabilities >= best_threshold
        ).astype(int)

        tuned_precision = precision_score(
            y_true,
            tuned_preds,
            zero_division=0
        )

        tuned_recall = recall_score(
            y_true,
            tuned_preds,
            zero_division=0
        )


    result = {
        "label":
            label,

        "seed":
            seed,

        "learning_rate":
            learning_rate,

        "config":
            config_name,

        "weight_decay":
            reg["weight_decay"],

        "warmup_ratio":
            reg["warmup_ratio"],

        "checkpoint_f1":
            checkpoint_f1,

        "default_precision":
            float(default_precision),

        "default_recall":
            float(default_recall),

        "default_f1":
            float(default_f1),

        "best_threshold":
            float(best_threshold),

        "tuned_precision":
            float(tuned_precision),

        "tuned_recall":
            float(tuned_recall),

        "tuned_f1":
            float(tuned_f1)
    }


    del trainer
    del model
    del output

    clear_memory()

    return result


# ============================================================
# STAGE A — LEARNING RATE SEARCH
# ============================================================

print("\n\n========================================")
print("STAGE A — DeBERTa LEARNING RATE SEARCH")
print("========================================")


baseline_winner = {}
best_lr = {}

lr_search_results = []


for label in LABELS:

    print(f"\n========== {label} ==========")

    candidates = []


    for lr in LEARNING_RATES:

        print(
            f"\n{label}"
            f" | LR={lr:.0e}"
            f" | baseline"
        )

        result = train_one(
            label=label,
            seed=42,
            learning_rate=lr,
            config_name="baseline"
        )

        candidates.append(result)
        lr_search_results.append(result)

        print(
            f"Default F1 = {result['default_f1']:.3f}"
        )

        print(
            f"Threshold  = {result['best_threshold']:.2f}"
        )

        print(
            f"Tuned F1   = {result['tuned_f1']:.3f}"
        )


    winner = max(
        candidates,
        key=lambda x: x["tuned_f1"]
    )

    baseline_winner[label] = winner

    best_lr[label] = winner[
        "learning_rate"
    ]

    print(
        f"\n>>> BEST LR {label}: "
        f"{best_lr[label]:.0e}"
    )


# ============================================================
# STAGE B — REGULARISATION SEARCH
# ============================================================

print("\n\n========================================")
print("STAGE B — DeBERTa REGULARISATION SEARCH")
print("========================================")


winning_seed42 = {}
best_config = {}


for label in LABELS:

    print(f"\n========== {label} ==========")

    candidates = [
        baseline_winner[label]
    ]


    for config_name in [
        "weight_decay",
        "weight_decay_warmup"
    ]:

        print(
            f"\n{label}"
            f" | LR={best_lr[label]:.0e}"
            f" | {config_name}"
        )

        result = train_one(
            label=label,
            seed=42,
            learning_rate=best_lr[label],
            config_name=config_name
        )

        candidates.append(result)

        print(
            f"Default F1 = {result['default_f1']:.3f}"
        )

        print(
            f"Threshold  = {result['best_threshold']:.2f}"
        )

        print(
            f"Tuned F1   = {result['tuned_f1']:.3f}"
        )


    winner = max(
        candidates,
        key=lambda x: x["tuned_f1"]
    )

    winning_seed42[label] = winner
    best_config[label] = winner["config"]

    print(
        f"\n>>> WINNER {label}: "
        f"{best_config[label]}"
        f" | LR={best_lr[label]:.0e}"
        f" | tuned F1={winner['tuned_f1']:.3f}"
    )


# ============================================================
# STAGE C — 3-SEED CONFIRMATION
# ============================================================

print("\n\n========================================")
print("STAGE C — DeBERTa 3-SEED CONFIRMATION")
print("========================================")


seed_results = []


for label in LABELS:

    print(f"\n========== {label} ==========")

    # Reuse seed 42 winner
    seed_results.append(
        winning_seed42[label]
    )


    for seed in [1, 2]:

        print(
            f"\n{label}"
            f" | seed={seed}"
            f" | LR={best_lr[label]:.0e}"
            f" | {best_config[label]}"
        )

        result = train_one(
            label=label,
            seed=seed,
            learning_rate=best_lr[label],
            config_name=best_config[label]
        )

        seed_results.append(result)

        print(
            f"Tuned F1 = {result['tuned_f1']:.3f}"
        )


# ============================================================
# STAGE D — FINAL SUMMARY
# ============================================================

results_df = pd.DataFrame(
    seed_results
)

summary_rows = []


print("\n\n========================================")
print("FINAL DeBERTa VALIDATION SUMMARY")
print("========================================")


for label in LABELS:

    subset = results_df[
        results_df["label"] == label
    ]

    summary = {
        "label":
            label,

        "best_learning_rate":
            best_lr[label],

        "best_config":
            best_config[label],

        "weight_decay":
            subset["weight_decay"].iloc[0],

        "warmup_ratio":
            subset["warmup_ratio"].iloc[0],

        "default_f1_mean":
            subset["default_f1"].mean(),

        "default_f1_sd":
            subset["default_f1"].std(ddof=0),

        "tuned_f1_mean":
            subset["tuned_f1"].mean(),

        "tuned_f1_sd":
            subset["tuned_f1"].std(ddof=0),

        "median_threshold":
            subset["best_threshold"].median()
    }


    summary_rows.append(
        summary
    )


    print(f"\n{label}")

    print(
        f"  Best LR: "
        f"{summary['best_learning_rate']:.0e}"
    )

    print(
        f"  Config: "
        f"{summary['best_config']}"
    )

    print(
        f"  Weight decay: "
        f"{summary['weight_decay']}"
    )

    print(
        f"  Warmup ratio: "
        f"{summary['warmup_ratio']}"
    )

    print(
        f"  Default F1: "
        f"{summary['default_f1_mean']:.3f}"
        f" ± {summary['default_f1_sd']:.3f}"
    )

    print(
        f"  Tuned F1: "
        f"{summary['tuned_f1_mean']:.3f}"
        f" ± {summary['tuned_f1_sd']:.3f}"
    )

    print(
        f"  Thresholds: "
        f"{[round(x, 2) for x in subset['best_threshold']]}"
    )

    print(
        f"  Median threshold: "
        f"{summary['median_threshold']:.2f}"
    )


summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# SAVE RESULTS
# ============================================================

results_path = (
    DATA_DIR
    / "results_deberta_finetuned_validation.csv"
)

summary_path = (
    DATA_DIR
    / "results_deberta_finetuned_summary.csv"
)


results_df.to_csv(
    results_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)


print("\nSaved:")
print(results_path)
print(summary_path)


# ============================================================
# DETAILED RESULTS
# ============================================================

display_df = results_df.copy()

display_df["learning_rate"] = (
    display_df["learning_rate"]
    .map(lambda x: f"{x:.0e}")
)


print("\n\n========================================")
print("DETAILED DeBERTa 3-SEED RESULTS")
print("========================================")


print(
    display_df[
        [
            "label",
            "seed",
            "learning_rate",
            "config",
            "weight_decay",
            "warmup_ratio",
            "checkpoint_f1",
            "default_precision",
            "default_recall",
            "default_f1",
            "best_threshold",
            "tuned_precision",
            "tuned_recall",
            "tuned_f1"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

## Hierarchical (nested-1B) experiments

Because 1B is a subtype of 1A, these development experiments test whether 1B is better predicted **directly** (text → 1B) or **hierarchically** (predict 1A first, then 1B only within grievance cases), and whether balanced class weighting over-corrects the rare 1B class. The outcome informed the final rule in notebook 10: 1B is predicted independently but **kept only where 1A is also positive** (the nesting constraint), rather than a trained two-stage cascade.

In [ ]:
# ============================================================
# Hierarchical vs direct 1B — development experiment (seed 42)
#
# 1B (weaponised victimhood) is a subtype of 1A (grievance).
# This compares two ways of predicting 1B:
#
#   DIRECT:        text -> 1B
#   HIERARCHICAL:  text -> 1A?  --(if yes)-->  weaponised?  -> 1B
#
# Two-stage training:
#   Stage 1: train on all train_fit rows        -> predict 1A
#   Stage 2: train only on gold (1A == 1) rows   -> predict 1B
#   Cascade: predicted_1B = predicted_1A AND predicted_weaponised
#
# Development only: validation is used for checkpoint selection and
# both threshold choices, so the validation F1 here is NOT an
# unbiased estimate. test.csv is never loaded.
#
# Inputs:   data/processed/train_fit.csv, validation.csv
# Optional: data/processed/results_deberta_finetuned_validation.csv
#           (if present, the matched seed-42 direct-1B precision/
#            recall/F1/threshold are pulled in for comparison)
# ============================================================

import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)


# ============================================================
# 2. SETTINGS
# ============================================================

MODEL_NAME = "microsoft/deberta-v3-base"

SEED = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

MODEL_DIR = Path(
    str(PROJECT_ROOT / "final_training_runs" / "hierarchical_deberta_models")
)

TRAIN_PATH = (
    DATA_DIR
    / "train_fit.csv"
)

VAL_PATH = (
    DATA_DIR
    / "validation.csv"
)

DIRECT_RESULTS_PATH = (
    DATA_DIR
    / "results_deberta_finetuned_validation.csv"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# WINNING DeBERTa SETTINGS
# ------------------------------------------------------------

# Stage 1 — 1A
LR_1A = 2e-5
WEIGHT_DECAY_1A = 0.00

# Stage 2 — use winning direct 1B settings
LR_WEAPON = 3e-5
WEIGHT_DECAY_WEAPON = 0.01


# ------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------

TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 2

MAX_LENGTH = 256
NUM_EPOCHS = 3

THRESHOLDS = np.arange(
    0.10,
    0.91,
    0.01
)


# ------------------------------------------------------------
# EXPECTED DEVELOPMENT SPLIT
#
# Prevent accidentally using different data from the
# direct DeBERTa benchmark.
# ------------------------------------------------------------

EXPECTED_TRAIN_ROWS = 1215
EXPECTED_VAL_ROWS = 405

EXPECTED_COUNTS = {

    "train_1A_positive": 377,
    "train_1B_positive": 118,

    "val_1A_positive": 126,
    "val_1B_positive": 40
}


# ------------------------------------------------------------
# FALLBACK DIRECT BENCHMARK
#
# Used only if previous results CSV is unavailable.
# Precision/recall are deliberately NOT invented.
# ------------------------------------------------------------

FALLBACK_DIRECT_F1 = 0.553

FALLBACK_DIRECT_THRESHOLD = np.nan
FALLBACK_DIRECT_PRECISION = np.nan
FALLBACK_DIRECT_RECALL = np.nan


# ============================================================
# 3. DEVELOPMENT WARNING
# ============================================================

print("=" * 72)

print(
    "HIERARCHICAL DeBERTa 1B "
    "— DEVELOPMENT EXPERIMENT"
)

print("=" * 72)

print(
    "\nThis cell does NOT read test.csv."
)

print(
    "Validation is being used for model development, "
    "checkpoint selection and threshold selection."
)

print(
    "The resulting validation F1 is therefore a "
    "DEVELOPMENT estimate only.\n"
)


# ============================================================
# 4. CHECK FILES
# ============================================================

for required_path in [
    TRAIN_PATH,
    VAL_PATH
]:

    if not required_path.exists():

        raise FileNotFoundError(
            f"Required file not found: "
            f"{required_path}"
        )


# ============================================================
# 5. LOAD DATA
# ============================================================

train_fit = pd.read_csv(
    TRAIN_PATH
)

val = pd.read_csv(
    VAL_PATH
)


required_columns = {
    "text_clean",
    "label_1A",
    "label_1B"
}


for name, df in [

    ("train_fit", train_fit),
    ("validation", val)

]:

    missing = (
        required_columns
        - set(df.columns)
    )

    if missing:

        raise ValueError(
            f"{name} is missing "
            f"required columns: {missing}"
        )


# ------------------------------------------------------------
# CLEAN TYPES
# ------------------------------------------------------------

for df in [
    train_fit,
    val
]:

    df["text_clean"] = (
        df["text_clean"]
        .fillna("")
        .astype(str)
    )

    df["label_1A"] = (
        df["label_1A"]
        .astype(int)
    )

    df["label_1B"] = (
        df["label_1B"]
        .astype(int)
    )


# ============================================================
# 6. VERIFY SAME DEVELOPMENT SPLIT
# ============================================================

if len(train_fit) != EXPECTED_TRAIN_ROWS:

    raise ValueError(
        f"train_fit has {len(train_fit)} rows, "
        f"expected {EXPECTED_TRAIN_ROWS}."
    )


if len(val) != EXPECTED_VAL_ROWS:

    raise ValueError(
        f"validation has {len(val)} rows, "
        f"expected {EXPECTED_VAL_ROWS}."
    )


actual_counts = {

    "train_1A_positive":
        int(
            train_fit[
                "label_1A"
            ].sum()
        ),

    "train_1B_positive":
        int(
            train_fit[
                "label_1B"
            ].sum()
        ),

    "val_1A_positive":
        int(
            val[
                "label_1A"
            ].sum()
        ),

    "val_1B_positive":
        int(
            val[
                "label_1B"
            ].sum()
        )
}


for key, expected in (
    EXPECTED_COUNTS.items()
):

    actual = actual_counts[key]

    if actual != expected:

        raise ValueError(
            f"{key} = {actual}, "
            f"expected {expected}. "
            "This does not appear to be "
            "the same development split."
        )


print("Development split verified.")

print(
    f"train_fit rows: "
    f"{len(train_fit)}"
)

print(
    f"validation rows: "
    f"{len(val)}"
)

print(
    f"train 1A positives: "
    f"{actual_counts['train_1A_positive']}"
)

print(
    f"train 1B positives: "
    f"{actual_counts['train_1B_positive']}"
)

print(
    f"validation 1A positives: "
    f"{actual_counts['val_1A_positive']}"
)

print(
    f"validation 1B positives: "
    f"{actual_counts['val_1B_positive']}"
)


# ============================================================
# 7. CHECK THE LABEL HIERARCHY
#
# 1B should NEVER exist without 1A.
# ============================================================

violations_train = train_fit[

    (train_fit["label_1B"] == 1)

    &

    (train_fit["label_1A"] == 0)
]


violations_val = val[

    (val["label_1B"] == 1)

    &

    (val["label_1A"] == 0)
]


print(
    "\nTraining hierarchy violations:",
    len(violations_train)
)

print(
    "Validation hierarchy violations:",
    len(violations_val)
)


if len(violations_train) > 0:

    raise ValueError(
        "Training data contains "
        "1B=1 where 1A=0."
    )


if len(violations_val) > 0:

    raise ValueError(
        "Validation data contains "
        "1B=1 where 1A=0."
    )


print(
    "Hierarchy check passed."
)


# ============================================================
# 8. LOAD MATCHED DIRECT DeBERTa BENCHMARK
#
# Prefer the actual saved seed-42 result.
#
# This allows:
#
#      DIRECT       vs      CASCADE
#
# Precision
# Recall
# F1
#
# using the same development split and seed.
# ============================================================

def load_direct_benchmark():

    fallback = {

        "precision":
            FALLBACK_DIRECT_PRECISION,

        "recall":
            FALLBACK_DIRECT_RECALL,

        "f1":
            FALLBACK_DIRECT_F1,

        "threshold":
            FALLBACK_DIRECT_THRESHOLD,

        "source":
            "known seed-42 F1 fallback"
    }


    if not DIRECT_RESULTS_PATH.exists():

        return fallback


    try:

        direct_df = pd.read_csv(
            DIRECT_RESULTS_PATH
        )

    except Exception as error:

        print(
            "\nCould not read direct "
            "DeBERTa results CSV:"
        )

        print(error)

        return fallback


    required = {
        "label",
        "seed",
        "tuned_f1"
    }


    if not required.issubset(
        direct_df.columns
    ):

        print(
            "\nDirect results file found, "
            "but expected columns are missing."
        )

        return fallback


    subset = direct_df[

        (
            direct_df[
                "label"
            ].astype(str)
            ==
            "label_1B"
        )

        &

        (
            pd.to_numeric(
                direct_df["seed"],
                errors="coerce"
            )
            ==
            SEED
        )

    ].copy()


    if len(subset) == 0:

        return fallback


    # --------------------------------------------------------
    # Prefer winning configuration:
    #
    # LR = 3e-5
    # weight_decay
    # --------------------------------------------------------

    if "config" in subset.columns:

        matching_config = subset[

            subset[
                "config"
            ].astype(str)
            ==
            "weight_decay"

        ]

        if len(matching_config) > 0:

            subset = matching_config


    if "learning_rate" in subset.columns:

        learning_rates = pd.to_numeric(
            subset[
                "learning_rate"
            ],
            errors="coerce"
        )

        matching_lr = subset[

            np.isclose(
                learning_rates,
                LR_WEAPON,
                rtol=0,
                atol=1e-10
            )

        ]

        if len(matching_lr) > 0:

            subset = matching_lr


    # If multiple remain, use highest tuned F1.
    tuned_f1_numeric = pd.to_numeric(
        subset[
            "tuned_f1"
        ],
        errors="coerce"
    )

    best_index = (
        tuned_f1_numeric
        .idxmax()
    )

    row = subset.loc[
        best_index
    ]


    result = {

        "precision":
            (
                float(
                    row[
                        "tuned_precision"
                    ]
                )
                if
                "tuned_precision"
                in row.index
                else np.nan
            ),

        "recall":
            (
                float(
                    row[
                        "tuned_recall"
                    ]
                )
                if
                "tuned_recall"
                in row.index
                else np.nan
            ),

        "f1":
            float(
                row[
                    "tuned_f1"
                ]
            ),

        "threshold":
            (
                float(
                    row[
                        "best_threshold"
                    ]
                )
                if
                "best_threshold"
                in row.index
                else np.nan
            ),

        "source":
            str(
                DIRECT_RESULTS_PATH
            )
    }


    return result


direct_benchmark = (
    load_direct_benchmark()
)


DIRECT_PRECISION = (
    direct_benchmark[
        "precision"
    ]
)

DIRECT_RECALL = (
    direct_benchmark[
        "recall"
    ]
)

DIRECT_F1 = (
    direct_benchmark[
        "f1"
    ]
)

DIRECT_THRESHOLD = (
    direct_benchmark[
        "threshold"
    ]
)


print("\n" + "=" * 72)
print("DIRECT DeBERTa BENCHMARK")
print("=" * 72)

print(
    "Source:",
    direct_benchmark[
        "source"
    ]
)

print(
    f"Seed: {SEED}"
)

if not np.isnan(
    DIRECT_PRECISION
):

    print(
        f"Precision: "
        f"{DIRECT_PRECISION:.3f}"
    )

else:

    print(
        "Precision: unavailable "
        "from fallback"
    )


if not np.isnan(
    DIRECT_RECALL
):

    print(
        f"Recall: "
        f"{DIRECT_RECALL:.3f}"
    )

else:

    print(
        "Recall: unavailable "
        "from fallback"
    )


print(
    f"F1: "
    f"{DIRECT_F1:.3f}"
)


if not np.isnan(
    DIRECT_THRESHOLD
):

    print(
        f"Threshold: "
        f"{DIRECT_THRESHOLD:.2f}"
    )


# ============================================================
# 9. CREATE STAGE-2 DATA
#
# Stage 2 learns ONLY:
#
#     grievance-only
#
#           vs
#
#     weaponised grievance
#
# using gold-labelled 1A-positive TRAINING examples.
# ============================================================

weapon_train = train_fit[

    train_fit[
        "label_1A"
    ]
    ==
    1

].copy()


weapon_val = val[

    val[
        "label_1A"
    ]
    ==
    1

].copy()


print("\n" + "=" * 72)
print("STAGE-2 DATA")
print("=" * 72)


print(
    f"Training grievance comments: "
    f"{len(weapon_train)}"
)


print(
    f"Weaponised grievance: "
    f"{int(weapon_train['label_1B'].sum())}"
)


print(
    f"Grievance-only: "
    f"{int(len(weapon_train) - weapon_train['label_1B'].sum())}"
)


print(
    f"\nValidation grievance comments: "
    f"{len(weapon_val)}"
)


print(
    f"Validation weaponised: "
    f"{int(weapon_val['label_1B'].sum())}"
)


print(
    f"Validation grievance-only: "
    f"{int(len(weapon_val) - weapon_val['label_1B'].sum())}"
)


# ============================================================
# 10. TOKENIZER
# ============================================================

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)


data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer
    )
)


# ============================================================
# 11. CREATE DATASETS
# ============================================================

def make_dataset(
    dataframe,
    label_column
):

    temp = dataframe[

        [
            "text_clean",
            label_column
        ]

    ].copy()


    temp = temp.rename(

        columns={
            label_column:
                "labels"
        }

    )


    dataset = (
        Dataset.from_pandas(
            temp,
            preserve_index=False
        )
    )


    def tokenize_batch(
        batch
    ):

        return tokenizer(

            batch[
                "text_clean"
            ],

            truncation=True,

            max_length=
                MAX_LENGTH
        )


    dataset = dataset.map(

        tokenize_batch,

        batched=True,

        remove_columns=[
            "text_clean"
        ]
    )


    return dataset


# ------------------------------------------------------------
# STAGE 1
# ------------------------------------------------------------

train_1A_ds = make_dataset(
    train_fit,
    "label_1A"
)

val_1A_ds = make_dataset(
    val,
    "label_1A"
)


# ------------------------------------------------------------
# STAGE 2 — GOLD 1A subset
# ------------------------------------------------------------

train_weapon_ds = make_dataset(
    weapon_train,
    "label_1B"
)

val_weapon_ds = make_dataset(
    weapon_val,
    "label_1B"
)


# ------------------------------------------------------------
# Stage 2 must ALSO predict all 405 validation rows so the
# final pipeline can be evaluated end-to-end.
# ------------------------------------------------------------

val_all_weapon_ds = make_dataset(
    val,
    "label_1B"
)


# ============================================================
# 12. METRICS
# ============================================================

def compute_metrics(
    eval_pred
):

    logits, labels = eval_pred


    if isinstance(
        logits,
        tuple
    ):

        logits = logits[0]


    predictions = np.argmax(
        logits,
        axis=1
    )


    return {

        "precision":
            precision_score(
                labels,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                predictions,
                zero_division=0
            )
    }


# ============================================================
# 13. GET POSITIVE-CLASS PROBABILITIES
# ============================================================

def get_positive_probabilities(
    logits
):

    if isinstance(
        logits,
        tuple
    ):

        logits = logits[0]


    logits_tensor = torch.tensor(

        logits,

        dtype=torch.float32

    )


    probabilities = torch.softmax(

        logits_tensor,

        dim=1

    ).numpy()


    return probabilities[
        :,
        1
    ]


# ============================================================
# 14. THRESHOLD SEARCH
#
# DEVELOPMENT ONLY.
# ============================================================

def find_best_threshold(
    y_true,
    probabilities
):

    best_threshold = 0.50
    best_score = -1.0


    for threshold in THRESHOLDS:

        predictions = (

            probabilities
            >=
            threshold

        ).astype(int)


        score = f1_score(

            y_true,
            predictions,

            zero_division=0

        )


        if score > best_score:

            best_score = float(
                score
            )

            best_threshold = float(
                threshold
            )


    return (
        best_threshold,
        best_score
    )


# ============================================================
# 15. MEMORY CLEANUP
# ============================================================

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 16. WEIGHTED TRAINER
# ============================================================

class WeightedTrainer(
    Trainer
):

    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs
    ):

        super().__init__(
            *args,
            **kwargs
        )

        self.class_weights = (
            class_weights
        )

        self.model_accepts_loss_kwargs = (
            False
        )


    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs
    ):

        labels = inputs[
            "labels"
        ]


        model_inputs = {

            key: value

            for key, value
            in inputs.items()

            if key != "labels"

        }


        outputs = model(
            **model_inputs
        )


        logits = (
            outputs.logits
            .float()
        )


        weights = (
            self.class_weights
            .to(
                device=
                    logits.device,

                dtype=
                    torch.float32
            )
        )


        loss = (
            nn.CrossEntropyLoss(
                weight=weights
            )(
                logits,
                labels
            )
        )


        if return_outputs:

            return (
                loss,
                outputs
            )


        return loss


# ============================================================
# 17. TRAIN MODEL FUNCTION
# ============================================================

def train_model(

    train_dataframe,

    train_dataset,

    eval_dataset,

    label_column,

    learning_rate,

    weight_decay,

    run_name,

    eval_steps

):

    clear_memory()

    set_seed(
        SEED
    )


    # --------------------------------------------------------
    # CLASS WEIGHTS
    # --------------------------------------------------------

    class_weights_np = (
        compute_class_weight(

            class_weight=
                "balanced",

            classes=
                np.array(
                    [0, 1]
                ),

            y=
                train_dataframe[
                    label_column
                ].to_numpy()
        )
    )


    class_weights = torch.tensor(

        class_weights_np,

        dtype=torch.float32

    )


    print(
        f"\n{run_name}"
    )

    print(
        "Class weights:",
        class_weights.tolist()
    )


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(

            MODEL_NAME,

            num_labels=2,

            dtype=torch.float32
        )
    )


    output_dir = (

        MODEL_DIR
        /
        run_name

    )


    if output_dir.exists():

        shutil.rmtree(
            output_dir
        )


    # --------------------------------------------------------
    # TRAINING ARGUMENTS
    #
    # save_strategy="best":
    # only save when validation F1 improves.
    #
    # This avoids the enormous repeated checkpoint writes
    # from the previous tuning runs.
    # --------------------------------------------------------

    training_args = (
        TrainingArguments(

            output_dir=
                str(output_dir),

            num_train_epochs=
                NUM_EPOCHS,

            per_device_train_batch_size=
                TRAIN_BATCH,

            per_device_eval_batch_size=
                EVAL_BATCH,

            gradient_accumulation_steps=
                GRAD_ACCUM,

            learning_rate=
                learning_rate,

            weight_decay=
                weight_decay,

            warmup_steps=
                0,

            eval_strategy=
                "steps",

            eval_steps=
                eval_steps,

            save_strategy=
                "best",

            load_best_model_at_end=
                True,

            metric_for_best_model=
                "f1",

            greater_is_better=
                True,

            save_total_limit=
                1,

            logging_steps=
                eval_steps,

            report_to=
                "none",

            seed=
                SEED,

            data_seed=
                SEED,

            fp16=
                False,

            bf16=
                False
        )
    )


    trainer = WeightedTrainer(

        model=
            model,

        args=
            training_args,

        train_dataset=
            train_dataset,

        eval_dataset=
            eval_dataset,

        data_collator=
            data_collator,

        compute_metrics=
            compute_metrics,

        class_weights=
            class_weights
    )


    trainer.train()


    return trainer


# ============================================================
# 18. STAGE 1
#
# ALL TRAINING COMMENTS -> 1A
#
# Winning DeBERTa:
# LR = 2e-5
# baseline
# ============================================================

print("\n\n" + "=" * 72)

print(
    "STAGE 1 — MALE GRIEVANCE / 1A"
)

print("=" * 72)


trainer_1A = train_model(

    train_dataframe=
        train_fit,

    train_dataset=
        train_1A_ds,

    eval_dataset=
        val_1A_ds,

    label_column=
        "label_1A",

    learning_rate=
        LR_1A,

    weight_decay=
        WEIGHT_DECAY_1A,

    run_name=
        "stage1_1A_seed42",

    eval_steps=
        50
)


# ============================================================
# 19. STAGE-1 VALIDATION PREDICTIONS
# ============================================================

stage1_output = (
    trainer_1A.predict(
        val_1A_ds
    )
)


prob_1A = (
    get_positive_probabilities(
        stage1_output.predictions
    )
)


true_1A = (
    val[
        "label_1A"
    ].to_numpy()
)


threshold_1A, stage1_f1 = (
    find_best_threshold(

        true_1A,
        prob_1A

    )
)


pred_1A = (

    prob_1A
    >=
    threshold_1A

).astype(int)


stage1_precision = (
    precision_score(

        true_1A,
        pred_1A,

        zero_division=0
    )
)


stage1_recall = (
    recall_score(

        true_1A,
        pred_1A,

        zero_division=0
    )
)


print(
    "\nStage 1 development result:"
)

print(
    f"Threshold = "
    f"{threshold_1A:.2f}"
)

print(
    f"Precision = "
    f"{stage1_precision:.3f}"
)

print(
    f"Recall = "
    f"{stage1_recall:.3f}"
)

print(
    f"F1 = "
    f"{stage1_f1:.3f}"
)


# ============================================================
# 20. STAGE 2
#
# GOLD 1A-POSITIVE TRAINING COMMENTS ONLY
#
# Task:
# grievance-only vs weaponised grievance
# ============================================================

print("\n\n" + "=" * 72)

print(
    "STAGE 2 — WEAPONISATION WITHIN GRIEVANCE"
)

print("=" * 72)


trainer_weapon = train_model(

    train_dataframe=
        weapon_train,

    train_dataset=
        train_weapon_ds,

    eval_dataset=
        val_weapon_ds,

    label_column=
        "label_1B",

    learning_rate=
        LR_WEAPON,

    weight_decay=
        WEIGHT_DECAY_WEAPON,

    run_name=
        "stage2_weaponisation_seed42",

    eval_steps=
        20
)


# ============================================================
# 21. CONDITIONAL STAGE-2 PERFORMANCE
#
# GOLD 1A-positive validation subset only.
#
# Diagnostic:
# Can the model distinguish grievance-only from
# weaponised grievance once grievance is known?
# ============================================================

conditional_output = (
    trainer_weapon.predict(
        val_weapon_ds
    )
)


prob_weapon_conditional = (
    get_positive_probabilities(

        conditional_output.predictions

    )
)


true_weapon_conditional = (
    weapon_val[
        "label_1B"
    ].to_numpy()
)


(
    conditional_threshold,
    conditional_f1

) = find_best_threshold(

    true_weapon_conditional,
    prob_weapon_conditional

)


conditional_pred = (

    prob_weapon_conditional
    >=
    conditional_threshold

).astype(int)


conditional_precision = (
    precision_score(

        true_weapon_conditional,
        conditional_pred,

        zero_division=0
    )
)


conditional_recall = (
    recall_score(

        true_weapon_conditional,
        conditional_pred,

        zero_division=0
    )
)


print(
    "\nConditional Stage-2 diagnostic:"
)

print(
    f"Threshold = "
    f"{conditional_threshold:.2f}"
)

print(
    f"Precision = "
    f"{conditional_precision:.3f}"
)

print(
    f"Recall = "
    f"{conditional_recall:.3f}"
)

print(
    f"F1 = "
    f"{conditional_f1:.3f}"
)


# ============================================================
# 22. APPLY STAGE 2 TO ALL VALIDATION ROWS
# ============================================================

stage2_all_output = (
    trainer_weapon.predict(
        val_all_weapon_ds
    )
)


prob_weapon_all = (
    get_positive_probabilities(

        stage2_all_output.predictions

    )
)


true_1B = (
    val[
        "label_1B"
    ].to_numpy()
)


# ============================================================
# 23. TUNE END-TO-END STAGE-2 THRESHOLD
#
# DEVELOPMENT ONLY.
#
# Stage-1 threshold is fixed above.
#
# Search Stage-2 threshold based on final 1B F1 across
# ALL 405 validation comments.
# ============================================================

best_stage2_threshold = 0.50
best_cascade_f1 = -1.0


for threshold in THRESHOLDS:

    stage2_pred = (

        prob_weapon_all
        >=
        threshold

    ).astype(int)


    cascade_pred = (

        (pred_1A == 1)

        &

        (stage2_pred == 1)

    ).astype(int)


    score = f1_score(

        true_1B,
        cascade_pred,

        zero_division=0
    )


    if score > best_cascade_f1:

        best_cascade_f1 = float(
            score
        )

        best_stage2_threshold = float(
            threshold
        )


# ============================================================
# 24. FINAL HARD CASCADE
# ============================================================

pred_weapon_all = (

    prob_weapon_all
    >=
    best_stage2_threshold

).astype(int)


cascade_pred_1B = (

    (pred_1A == 1)

    &

    (pred_weapon_all == 1)

).astype(int)


cascade_precision = (
    precision_score(

        true_1B,
        cascade_pred_1B,

        zero_division=0
    )
)


cascade_recall = (
    recall_score(

        true_1B,
        cascade_pred_1B,

        zero_division=0
    )
)


cascade_f1 = (
    f1_score(

        true_1B,
        cascade_pred_1B,

        zero_division=0
    )
)


cascade_cm = (
    confusion_matrix(

        true_1B,
        cascade_pred_1B,

        labels=[
            0,
            1
        ]
    )
)


tn, fp, fn, tp = (
    cascade_cm.ravel()
)


# ============================================================
# 25. ORACLE-GATE DIAGNOSTIC
#
# Replace predicted Stage-1 status with GOLD 1A labels.
#
# This answers:
#
# If grievance detection were perfect, how good would
# Stage 2 be in the complete 1B task?
#
# This is DIAGNOSTIC ONLY.
# ============================================================

oracle_gate_pred = (

    (true_1A == 1)

    &

    (pred_weapon_all == 1)

).astype(int)


oracle_precision = (
    precision_score(

        true_1B,
        oracle_gate_pred,

        zero_division=0
    )
)


oracle_recall = (
    recall_score(

        true_1B,
        oracle_gate_pred,

        zero_division=0
    )
)


oracle_f1 = (
    f1_score(

        true_1B,
        oracle_gate_pred,

        zero_division=0
    )
)


# ============================================================
# 26. ERROR PROPAGATION
#
# How many true 1B comments did Stage 1 remove before they
# ever reached the weaponisation decision?
# ============================================================

true_1B_mask = (
    true_1B
    ==
    1
)


true_1B_lost_at_stage1 = int(

    np.sum(

        true_1B_mask

        &

        (pred_1A == 0)

    )
)


true_1B_survived_stage1 = int(

    np.sum(

        true_1B_mask

        &

        (pred_1A == 1)

    )
)


stage1_survival_rate = (

    true_1B_survived_stage1
    /
    int(
        true_1B.sum()
    )

)


# ============================================================
# 27. DIRECT VS HIERARCHICAL
# ============================================================

f1_difference = (
    cascade_f1
    -
    DIRECT_F1
)


if not np.isnan(
    DIRECT_PRECISION
):

    precision_difference = (

        cascade_precision
        -
        DIRECT_PRECISION

    )

else:

    precision_difference = np.nan


if not np.isnan(
    DIRECT_RECALL
):

    recall_difference = (

        cascade_recall
        -
        DIRECT_RECALL

    )

else:

    recall_difference = np.nan


# ============================================================
# 28. PRINT FINAL DEVELOPMENT RESULTS
# ============================================================

print("\n\n" + "=" * 72)

print(
    "HIERARCHICAL 1B — DEVELOPMENT RESULTS"
)

print("=" * 72)


print(
    "\nSTAGE 1 — GRIEVANCE"
)

print(
    f"Threshold: "
    f"{threshold_1A:.2f}"
)

print(
    f"Precision: "
    f"{stage1_precision:.3f}"
)

print(
    f"Recall: "
    f"{stage1_recall:.3f}"
)

print(
    f"F1: "
    f"{stage1_f1:.3f}"
)


print(
    "\nSTAGE 2 — GOLD GRIEVANCE SUBSET"
)

print(
    f"Threshold: "
    f"{conditional_threshold:.2f}"
)

print(
    f"Precision: "
    f"{conditional_precision:.3f}"
)

print(
    f"Recall: "
    f"{conditional_recall:.3f}"
)

print(
    f"F1: "
    f"{conditional_f1:.3f}"
)


print(
    "\nEND-TO-END HARD CASCADE"
)

print(
    f"Stage-2 threshold: "
    f"{best_stage2_threshold:.2f}"
)

print(
    f"Precision: "
    f"{cascade_precision:.3f}"
)

print(
    f"Recall: "
    f"{cascade_recall:.3f}"
)

print(
    f"F1: "
    f"{cascade_f1:.3f}"
)


print(
    "\nConfusion matrix:"
)

print(
    f"TN={tn} | FP={fp}"
)

print(
    f"FN={fn} | TP={tp}"
)


print(
    "\nERROR PROPAGATION"
)

print(
    f"True 1B cases total: "
    f"{int(true_1B.sum())}"
)

print(
    f"True 1B surviving Stage 1: "
    f"{true_1B_survived_stage1}"
)

print(
    f"True 1B lost at Stage 1: "
    f"{true_1B_lost_at_stage1}"
)

print(
    f"Stage-1 survival rate for true 1B: "
    f"{stage1_survival_rate:.3f}"
)


print(
    "\nORACLE-GATE DIAGNOSTIC"
)

print(
    f"Precision: "
    f"{oracle_precision:.3f}"
)

print(
    f"Recall: "
    f"{oracle_recall:.3f}"
)

print(
    f"F1: "
    f"{oracle_f1:.3f}"
)


print("\n" + "-" * 72)

print(
    "DIRECT vs HIERARCHICAL "
    "— SAME DEVELOPMENT SPLIT / SEED"
)

print("-" * 72)


if not np.isnan(
    DIRECT_PRECISION
):

    print(
        f"Direct precision:      "
        f"{DIRECT_PRECISION:.3f}"
    )

else:

    print(
        "Direct precision:      "
        "not available"
    )


print(
    f"Hierarchical precision:"
    f" {cascade_precision:.3f}"
)


if not np.isnan(
    precision_difference
):

    print(
        f"Precision change:      "
        f"{precision_difference:+.3f}"
    )


print()


if not np.isnan(
    DIRECT_RECALL
):

    print(
        f"Direct recall:         "
        f"{DIRECT_RECALL:.3f}"
    )

else:

    print(
        "Direct recall:         "
        "not available"
    )


print(
    f"Hierarchical recall:   "
    f" {cascade_recall:.3f}"
)


if not np.isnan(
    recall_difference
):

    print(
        f"Recall change:         "
        f"{recall_difference:+.3f}"
    )


print()


print(
    f"Direct F1:             "
    f"{DIRECT_F1:.3f}"
)

print(
    f"Hierarchical F1:       "
    f"{cascade_f1:.3f}"
)

print(
    f"F1 change:             "
    f"{f1_difference:+.3f}"
)


print("-" * 72)


# ============================================================
# 29. AUTOMATIC INTERPRETATION
# ============================================================

print(
    "\nDEVELOPMENT INTERPRETATION:"
)


if cascade_f1 > DIRECT_F1:

    print(
        "✓ The hierarchical hard cascade "
        "improved seed-42 end-to-end 1B F1."
    )

else:

    print(
        "✗ The hierarchical hard cascade "
        "did not improve seed-42 end-to-end 1B F1."
    )


oracle_gap = (
    oracle_f1
    -
    cascade_f1
)


print(
    f"\nOracle-minus-cascade F1 gap: "
    f"{oracle_gap:+.3f}"
)


if oracle_gap >= 0.05:

    print(
        "The oracle gate is substantially better than "
        "the real cascade."
    )

    print(
        "This suggests Stage-1 grievance errors / "
        "hard gating are an important bottleneck."
    )

    print(
        "A soft probability-composition experiment "
        "may therefore be worth testing next."
    )

else:

    print(
        "The oracle gate is fairly close to the "
        "real cascade."
    )

    print(
        "This suggests Stage 2 itself is likely a "
        "major limiting factor."
    )


print(
    "\nREMINDER:"
)

print(
    "These are DEVELOPMENT results."
)

print(
    "Do not evaluate test.csv until the final "
    "model strategy and thresholds are frozen."
)


# ============================================================
# 30. SAVE ROW-LEVEL PREDICTIONS
#
# Useful for manual error analysis.
# ============================================================

predictions_df = (
    val.copy()
)


predictions_df[
    "hier_prob_1A"
] = prob_1A


predictions_df[
    "hier_pred_1A"
] = pred_1A


predictions_df[
    "hier_prob_weaponisation"
] = prob_weapon_all


predictions_df[
    "hier_pred_weaponisation"
] = pred_weapon_all


predictions_df[
    "hier_pred_1B"
] = cascade_pred_1B


predictions_df[
    "oracle_pred_1B"
] = oracle_gate_pred


predictions_df[
    "hier_1B_error"
] = np.select(

    [

        (
            (true_1B == 1)
            &
            (cascade_pred_1B == 1)
        ),

        (
            (true_1B == 0)
            &
            (cascade_pred_1B == 1)
        ),

        (
            (true_1B == 1)
            &
            (cascade_pred_1B == 0)
        )

    ],

    [
        "TP",
        "FP",
        "FN"
    ],

    default=
        "TN"
)


predictions_df[
    "true_1B_lost_at_stage1"
] = (

    (
        (true_1B == 1)
        &
        (pred_1A == 0)
    )

    .astype(int)

)


predictions_path = (

    DATA_DIR
    /
    "results_deberta_hierarchical_1B_predictions_seed42.csv"

)


predictions_df.to_csv(

    predictions_path,

    index=False

)


# ============================================================
# 31. SAVE SUMMARY
# ============================================================

summary_df = pd.DataFrame(

    [

        {

            "development_only":
                True,

            "seed":
                SEED,

            "model":
                MODEL_NAME,

            "train_rows":
                len(train_fit),

            "validation_rows":
                len(val),

            # -------------------------
            # Stage 1
            # -------------------------

            "stage1_lr":
                LR_1A,

            "stage1_weight_decay":
                WEIGHT_DECAY_1A,

            "stage1_threshold":
                threshold_1A,

            "stage1_precision":
                stage1_precision,

            "stage1_recall":
                stage1_recall,

            "stage1_f1":
                stage1_f1,

            # -------------------------
            # Stage 2
            # -------------------------

            "stage2_training_rows":
                len(
                    weapon_train
                ),

            "stage2_positive_rows":
                int(
                    weapon_train[
                        "label_1B"
                    ].sum()
                ),

            "stage2_negative_rows":
                int(

                    len(
                        weapon_train
                    )

                    -

                    weapon_train[
                        "label_1B"
                    ].sum()

                ),

            "stage2_lr":
                LR_WEAPON,

            "stage2_weight_decay":
                WEIGHT_DECAY_WEAPON,

            "conditional_threshold":
                conditional_threshold,

            "conditional_precision":
                conditional_precision,

            "conditional_recall":
                conditional_recall,

            "conditional_f1":
                conditional_f1,

            # -------------------------
            # Cascade
            # -------------------------

            "cascade_stage2_threshold":
                best_stage2_threshold,

            "cascade_precision":
                cascade_precision,

            "cascade_recall":
                cascade_recall,

            "cascade_f1":
                cascade_f1,

            "cascade_tn":
                tn,

            "cascade_fp":
                fp,

            "cascade_fn":
                fn,

            "cascade_tp":
                tp,

            # -------------------------
            # Oracle
            # -------------------------

            "oracle_precision":
                oracle_precision,

            "oracle_recall":
                oracle_recall,

            "oracle_f1":
                oracle_f1,

            "oracle_minus_cascade_f1":
                oracle_gap,

            # -------------------------
            # Error propagation
            # -------------------------

            "true_1B_total":
                int(
                    true_1B.sum()
                ),

            "true_1B_survived_stage1":
                true_1B_survived_stage1,

            "true_1B_lost_at_stage1":
                true_1B_lost_at_stage1,

            "true_1B_stage1_survival_rate":
                stage1_survival_rate,

            # -------------------------
            # Direct benchmark
            # -------------------------

            "direct_precision":
                DIRECT_PRECISION,

            "direct_recall":
                DIRECT_RECALL,

            "direct_f1":
                DIRECT_F1,

            "direct_threshold":
                DIRECT_THRESHOLD,

            "direct_benchmark_source":
                direct_benchmark[
                    "source"
                ],

            # -------------------------
            # Difference
            # -------------------------

            "hierarchical_minus_direct_precision":
                precision_difference,

            "hierarchical_minus_direct_recall":
                recall_difference,

            "hierarchical_minus_direct_f1":
                f1_difference
        }

    ]

)


summary_path = (

    DATA_DIR
    /
    "results_deberta_hierarchical_1B_seed42.csv"

)


summary_df.to_csv(

    summary_path,

    index=False

)


# ============================================================
# 32. SAVED FILES
# ============================================================

print("\nSaved:")

print(
    predictions_path
)

print(
    summary_path
)


# ============================================================
# 33. CLEANUP
# ============================================================

del trainer_1A
del trainer_weapon

clear_memory()


print(
    "\nExperiment complete."
)

In [ ]:
# ============================================================
# Hierarchical 1B — unweighted Stage-2 ablation (seed 42, dev only)
#
# Follow-up to the hierarchical experiment above, whose results were:
#   Stage 2 (class-weighted): P = 0.325  R = 1.000  F1 = 0.491
#   Hard cascade:             P = 0.233  R = 0.925  F1 = 0.372
#   Direct DeBERTa benchmark: F1 = 0.553
#
# Question: did balanced class weighting over-correct Stage 2
# (perfect recall, weak precision)? This keeps the hierarchical
# formulation but trains Stage 2 with plain, unweighted
# cross-entropy; Stage 1 is the same weighted 1A model (reused from
# checkpoint if available).
#
# Development only. Inputs: train_fit.csv, validation.csv.
# test.csv is never loaded.
# ============================================================

import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)


# ============================================================
# 2. SETTINGS
# ============================================================

MODEL_NAME = "microsoft/deberta-v3-base"

SEED = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = DATA_DIR / "train_fit.csv"
VAL_PATH = DATA_DIR / "validation.csv"

MODEL_ROOT = Path(
    str(PROJECT_ROOT / "final_training_runs" / "hierarchical_deberta_models")
)

# Previous Stage-1 location
STAGE1_DIR = (
    MODEL_ROOT
    / "stage1_1A_seed42"
)

# New Stage-2 location
STAGE2_DIR = (
    MODEL_ROOT
    / "stage2_weaponisation_UNWEIGHTED_seed42"
)

DIRECT_RESULTS_PATH = (
    DATA_DIR
    / "results_deberta_finetuned_validation.csv"
)


# ------------------------------------------------------------
# Winning DeBERTa settings
# ------------------------------------------------------------

# Stage 1 — grievance
LR_1A = 2e-5
WEIGHT_DECAY_1A = 0.00

# Stage 2 — keep same LR / WD as previous experiment
# Change ONLY the loss weighting.
LR_STAGE2 = 3e-5
WEIGHT_DECAY_STAGE2 = 0.01


# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 2

MAX_LENGTH = 256
NUM_EPOCHS = 3

THRESHOLDS = np.arange(
    0.10,
    0.91,
    0.01
)


# ------------------------------------------------------------
# Expected split
# ------------------------------------------------------------

EXPECTED_TRAIN_ROWS = 1215
EXPECTED_VAL_ROWS = 405

EXPECTED_COUNTS = {
    "train_1A": 377,
    "train_1B": 118,
    "val_1A": 126,
    "val_1B": 40
}


# ------------------------------------------------------------
# Direct benchmark fallback
# ------------------------------------------------------------

DIRECT_F1_FALLBACK = 0.553


# ============================================================
# 3. WARNING
# ============================================================

print("=" * 72)

print(
    "HIERARCHICAL DeBERTa — "
    "UNWEIGHTED STAGE-2 ABLATION"
)

print("=" * 72)

print(
    "\nDEVELOPMENT ONLY."
)

print(
    "test.csv is NOT used."
)

print(
    "The only intended substantive change is:"
)

print(
    "Stage 2 now uses ordinary UNWEIGHTED "
    "cross-entropy.\n"
)


# ============================================================
# 4. LOAD DATA
# ============================================================

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Missing: {TRAIN_PATH}"
    )

if not VAL_PATH.exists():
    raise FileNotFoundError(
        f"Missing: {VAL_PATH}"
    )


train_fit = pd.read_csv(
    TRAIN_PATH
)

val = pd.read_csv(
    VAL_PATH
)


required_columns = {
    "text_clean",
    "label_1A",
    "label_1B"
}


for name, df in [
    ("train_fit", train_fit),
    ("validation", val)
]:

    missing = (
        required_columns
        - set(df.columns)
    )

    if missing:

        raise ValueError(
            f"{name} missing columns: "
            f"{missing}"
        )


for df in [
    train_fit,
    val
]:

    df["text_clean"] = (
        df["text_clean"]
        .fillna("")
        .astype(str)
    )

    df["label_1A"] = (
        df["label_1A"]
        .astype(int)
    )

    df["label_1B"] = (
        df["label_1B"]
        .astype(int)
    )


# ============================================================
# 5. VERIFY SPLIT
# ============================================================

if len(train_fit) != EXPECTED_TRAIN_ROWS:

    raise ValueError(
        f"Expected {EXPECTED_TRAIN_ROWS} "
        f"training rows; found "
        f"{len(train_fit)}."
    )


if len(val) != EXPECTED_VAL_ROWS:

    raise ValueError(
        f"Expected {EXPECTED_VAL_ROWS} "
        f"validation rows; found "
        f"{len(val)}."
    )


actual_counts = {

    "train_1A":
        int(
            train_fit[
                "label_1A"
            ].sum()
        ),

    "train_1B":
        int(
            train_fit[
                "label_1B"
            ].sum()
        ),

    "val_1A":
        int(
            val[
                "label_1A"
            ].sum()
        ),

    "val_1B":
        int(
            val[
                "label_1B"
            ].sum()
        )
}


for key, expected in (
    EXPECTED_COUNTS.items()
):

    if (
        actual_counts[key]
        !=
        expected
    ):

        raise ValueError(
            f"{key}: expected "
            f"{expected}, found "
            f"{actual_counts[key]}."
        )


print(
    "Development split verified."
)

print(
    f"train_fit = {len(train_fit)}"
)

print(
    f"validation = {len(val)}"
)

print(
    f"Train 1A positives = "
    f"{actual_counts['train_1A']}"
)

print(
    f"Train 1B positives = "
    f"{actual_counts['train_1B']}"
)

print(
    f"Validation 1A positives = "
    f"{actual_counts['val_1A']}"
)

print(
    f"Validation 1B positives = "
    f"{actual_counts['val_1B']}"
)


# ============================================================
# 6. HIERARCHY CHECK
# ============================================================

violations_train = train_fit[
    (train_fit["label_1B"] == 1)
    &
    (train_fit["label_1A"] == 0)
]


violations_val = val[
    (val["label_1B"] == 1)
    &
    (val["label_1A"] == 0)
]


if len(violations_train) > 0:

    raise ValueError(
        "Training data contains "
        "1B=1 where 1A=0."
    )


if len(violations_val) > 0:

    raise ValueError(
        "Validation data contains "
        "1B=1 where 1A=0."
    )


print(
    "\nHierarchy check passed."
)


# ============================================================
# 7. STAGE-2 SUBSET
# ============================================================

weapon_train = train_fit[
    train_fit[
        "label_1A"
    ] == 1
].copy()


weapon_val = val[
    val[
        "label_1A"
    ] == 1
].copy()


n_stage2 = len(
    weapon_train
)

n_positive = int(
    weapon_train[
        "label_1B"
    ].sum()
)

n_negative = (
    n_stage2
    -
    n_positive
)

positive_rate = (
    n_positive
    /
    n_stage2
)


print(
    "\n" + "=" * 72
)

print(
    "STAGE-2 TRAINING DATA"
)

print(
    "=" * 72
)

print(
    f"Total grievance comments: "
    f"{n_stage2}"
)

print(
    f"Weaponised: "
    f"{n_positive}"
)

print(
    f"Grievance-only: "
    f"{n_negative}"
)

print(
    f"Positive proportion: "
    f"{positive_rate:.3f}"
)

print(
    "\nStage 2 will use "
    "UNWEIGHTED cross-entropy."
)


# ============================================================
# 8. TOKENIZER
# ============================================================

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)


data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer
    )
)


# ============================================================
# 9. DATASET FUNCTION
# ============================================================

def make_dataset(
    dataframe,
    label_column
):

    temp = dataframe[
        [
            "text_clean",
            label_column
        ]
    ].copy()


    temp = temp.rename(
        columns={
            label_column:
                "labels"
        }
    )


    dataset = Dataset.from_pandas(
        temp,
        preserve_index=False
    )


    dataset = dataset.map(

        lambda batch: tokenizer(

            batch[
                "text_clean"
            ],

            truncation=True,

            max_length=
                MAX_LENGTH
        ),

        batched=True,

        remove_columns=[
            "text_clean"
        ]
    )


    return dataset


# Stage 1
train_1A_ds = make_dataset(
    train_fit,
    "label_1A"
)

val_1A_ds = make_dataset(
    val,
    "label_1A"
)


# Stage 2
train_weapon_ds = make_dataset(
    weapon_train,
    "label_1B"
)

val_weapon_ds = make_dataset(
    weapon_val,
    "label_1B"
)


# Stage 2 across ALL validation rows
val_all_weapon_ds = make_dataset(
    val,
    "label_1B"
)


# ============================================================
# 10. METRICS
# ============================================================

def compute_metrics(
    eval_pred
):

    logits, labels = (
        eval_pred
    )


    if isinstance(
        logits,
        tuple
    ):

        logits = logits[0]


    predictions = np.argmax(
        logits,
        axis=1
    )


    return {

        "precision":
            precision_score(
                labels,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                predictions,
                zero_division=0
            )
    }


# ============================================================
# 11. PROBABILITIES
# ============================================================

def positive_probabilities(
    logits
):

    if isinstance(
        logits,
        tuple
    ):

        logits = logits[0]


    tensor = torch.tensor(
        logits,
        dtype=torch.float32
    )


    probs = torch.softmax(
        tensor,
        dim=1
    ).numpy()


    return probs[:, 1]


# ============================================================
# 12. THRESHOLD SEARCH
# ============================================================

def best_threshold(
    y_true,
    probabilities
):

    best_t = 0.50
    best_f1 = -1.0


    for threshold in (
        THRESHOLDS
    ):

        predictions = (
            probabilities
            >=
            threshold
        ).astype(int)


        score = f1_score(
            y_true,
            predictions,
            zero_division=0
        )


        if score > best_f1:

            best_f1 = float(
                score
            )

            best_t = float(
                threshold
            )


    return (
        best_t,
        best_f1
    )


# ============================================================
# 13. CLEAN MEMORY
# ============================================================

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 14. WEIGHTED TRAINER — STAGE 1 ONLY
#
# Stage 1 stays EXACTLY conceptually as before.
# ============================================================

class WeightedTrainer(
    Trainer
):

    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs
    ):

        super().__init__(
            *args,
            **kwargs
        )

        self.class_weights = (
            class_weights
        )

        self.model_accepts_loss_kwargs = (
            False
        )


    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs
    ):

        labels = inputs[
            "labels"
        ]


        model_inputs = {

            key: value

            for key, value
            in inputs.items()

            if key != "labels"

        }


        outputs = model(
            **model_inputs
        )


        logits = (
            outputs.logits
            .float()
        )


        weights = (
            self.class_weights
            .to(
                device=
                    logits.device,

                dtype=
                    torch.float32
            )
        )


        loss = nn.CrossEntropyLoss(
            weight=weights
        )(
            logits,
            labels
        )


        if return_outputs:

            return (
                loss,
                outputs
            )


        return loss


# ============================================================
# 15. FIND PREVIOUS STAGE-1 CHECKPOINT
# ============================================================

def find_saved_checkpoint(
    run_directory
):

    if not run_directory.exists():

        return None


    checkpoints = list(
        run_directory.glob(
            "checkpoint-*"
        )
    )


    if len(checkpoints) == 0:

        return None


    checkpoints = sorted(

        checkpoints,

        key=lambda path:
            int(
                path.name.split(
                    "-"
                )[-1]
            )

    )


    # save_total_limit=1 should normally
    # leave only the best checkpoint.
    return checkpoints[-1]


stage1_checkpoint = (
    find_saved_checkpoint(
        STAGE1_DIR
    )
)


# ============================================================
# 16. STAGE 1
#
# REUSE previous checkpoint if available.
#
# Otherwise retrain the same Stage-1 model.
# ============================================================

clear_memory()

set_seed(
    SEED
)


if stage1_checkpoint is not None:

    print(
        "\n" + "=" * 72
    )

    print(
        "STAGE 1 — REUSING SAVED MODEL"
    )

    print(
        "=" * 72
    )

    print(
        f"Checkpoint: "
        f"{stage1_checkpoint}"
    )


    stage1_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            stage1_checkpoint,
            dtype=torch.float32
        )
    )


    stage1_predict_args = (
        TrainingArguments(

            output_dir=
                str(PROJECT_ROOT / "final_training_runs" / "stage1_prediction_tmp"),

            per_device_eval_batch_size=
                EVAL_BATCH,

            report_to=
                "none",

            fp16=
                False,

            bf16=
                False
        )
    )


    trainer_1A = Trainer(

        model=
            stage1_model,

        args=
            stage1_predict_args,

        data_collator=
            data_collator
    )


else:

    print(
        "\n" + "=" * 72
    )

    print(
        "NO SAVED STAGE-1 CHECKPOINT FOUND"
    )

    print(
        "Retraining Stage 1..."
    )

    print(
        "=" * 72
    )


    stage1_weights_np = (
        compute_class_weight(

            class_weight=
                "balanced",

            classes=
                np.array(
                    [0, 1]
                ),

            y=
                train_fit[
                    "label_1A"
                ].to_numpy()
        )
    )


    stage1_weights = (
        torch.tensor(

            stage1_weights_np,

            dtype=torch.float32

        )
    )


    stage1_model = (
        AutoModelForSequenceClassification
        .from_pretrained(

            MODEL_NAME,

            num_labels=2,

            dtype=torch.float32
        )
    )


    if STAGE1_DIR.exists():

        shutil.rmtree(
            STAGE1_DIR
        )


    stage1_args = (
        TrainingArguments(

            output_dir=
                str(
                    STAGE1_DIR
                ),

            num_train_epochs=
                NUM_EPOCHS,

            per_device_train_batch_size=
                TRAIN_BATCH,

            per_device_eval_batch_size=
                EVAL_BATCH,

            gradient_accumulation_steps=
                GRAD_ACCUM,

            learning_rate=
                LR_1A,

            weight_decay=
                WEIGHT_DECAY_1A,

            warmup_steps=
                0,

            eval_strategy=
                "steps",

            eval_steps=
                50,

            save_strategy=
                "best",

            load_best_model_at_end=
                True,

            metric_for_best_model=
                "f1",

            greater_is_better=
                True,

            save_total_limit=
                1,

            logging_steps=
                50,

            report_to=
                "none",

            seed=
                SEED,

            data_seed=
                SEED,

            fp16=
                False,

            bf16=
                False
        )
    )


    trainer_1A = WeightedTrainer(

        model=
            stage1_model,

        args=
            stage1_args,

        train_dataset=
            train_1A_ds,

        eval_dataset=
            val_1A_ds,

        data_collator=
            data_collator,

        compute_metrics=
            compute_metrics,

        class_weights=
            stage1_weights
    )


    trainer_1A.train()


# ============================================================
# 17. STAGE-1 VALIDATION RESULT
# ============================================================

stage1_output = (
    trainer_1A.predict(
        val_1A_ds
    )
)


prob_1A = (
    positive_probabilities(
        stage1_output.predictions
    )
)


true_1A = (
    val[
        "label_1A"
    ].to_numpy()
)


(
    threshold_1A,
    stage1_f1

) = best_threshold(

    true_1A,
    prob_1A

)


pred_1A = (
    prob_1A
    >=
    threshold_1A
).astype(int)


stage1_precision = (
    precision_score(
        true_1A,
        pred_1A,
        zero_division=0
    )
)


stage1_recall = (
    recall_score(
        true_1A,
        pred_1A,
        zero_division=0
    )
)


print(
    "\nStage 1:"
)

print(
    f"Threshold = "
    f"{threshold_1A:.2f}"
)

print(
    f"Precision = "
    f"{stage1_precision:.3f}"
)

print(
    f"Recall = "
    f"{stage1_recall:.3f}"
)

print(
    f"F1 = "
    f"{stage1_f1:.3f}"
)


# ============================================================
# 18. REMOVE STAGE-1 MODEL FROM GPU
#
# Predictions are saved already.
# ============================================================

del trainer_1A
del stage1_model

clear_memory()


# ============================================================
# 19. STAGE 2 — UNWEIGHTED
#
# THIS IS THE ACTUAL EXPERIMENTAL CHANGE.
#
# Standard Trainer means the model uses its ordinary
# unweighted classification loss.
#
# NO compute_class_weight()
# NO WeightedTrainer
# ============================================================

print(
    "\n\n" + "=" * 72
)

print(
    "STAGE 2 — UNWEIGHTED "
    "WEAPONISATION CLASSIFIER"
)

print(
    "=" * 72
)


set_seed(
    SEED
)


stage2_model = (
    AutoModelForSequenceClassification
    .from_pretrained(

        MODEL_NAME,

        num_labels=2,

        dtype=torch.float32
    )
)


if STAGE2_DIR.exists():

    shutil.rmtree(
        STAGE2_DIR
    )


stage2_args = (
    TrainingArguments(

        output_dir=
            str(
                STAGE2_DIR
            ),

        num_train_epochs=
            NUM_EPOCHS,

        per_device_train_batch_size=
            TRAIN_BATCH,

        per_device_eval_batch_size=
            EVAL_BATCH,

        gradient_accumulation_steps=
            GRAD_ACCUM,

        learning_rate=
            LR_STAGE2,

        weight_decay=
            WEIGHT_DECAY_STAGE2,

        warmup_steps=
            0,

        eval_strategy=
            "steps",

        # smaller training subset
        eval_steps=
            20,

        save_strategy=
            "best",

        load_best_model_at_end=
            True,

        metric_for_best_model=
            "f1",

        greater_is_better=
            True,

        save_total_limit=
            1,

        logging_steps=
            20,

        report_to=
            "none",

        seed=
            SEED,

        data_seed=
            SEED,

        fp16=
            False,

        bf16=
            False
    )
)


# ------------------------------------------------------------
# IMPORTANT:
# Plain Trainer = ordinary unweighted CE.
# ------------------------------------------------------------

trainer_stage2 = Trainer(

    model=
        stage2_model,

    args=
        stage2_args,

    train_dataset=
        train_weapon_ds,

    eval_dataset=
        val_weapon_ds,

    data_collator=
        data_collator,

    compute_metrics=
        compute_metrics
)


trainer_stage2.train()


# ============================================================
# 20. CONDITIONAL STAGE-2 PERFORMANCE
#
# GOLD grievance subset.
# ============================================================

conditional_output = (
    trainer_stage2.predict(
        val_weapon_ds
    )
)


prob_weapon_conditional = (
    positive_probabilities(
        conditional_output.predictions
    )
)


true_weapon_conditional = (
    weapon_val[
        "label_1B"
    ].to_numpy()
)


(
    conditional_threshold,
    conditional_f1

) = best_threshold(

    true_weapon_conditional,
    prob_weapon_conditional

)


conditional_pred = (
    prob_weapon_conditional
    >=
    conditional_threshold
).astype(int)


conditional_precision = (
    precision_score(
        true_weapon_conditional,
        conditional_pred,
        zero_division=0
    )
)


conditional_recall = (
    recall_score(
        true_weapon_conditional,
        conditional_pred,
        zero_division=0
    )
)


print(
    "\nUNWEIGHTED Stage-2 "
    "conditional result:"
)

print(
    f"Threshold = "
    f"{conditional_threshold:.2f}"
)

print(
    f"Precision = "
    f"{conditional_precision:.3f}"
)

print(
    f"Recall = "
    f"{conditional_recall:.3f}"
)

print(
    f"F1 = "
    f"{conditional_f1:.3f}"
)


# ============================================================
# 21. APPLY STAGE 2 TO ALL VALIDATION ROWS
# ============================================================

stage2_all_output = (
    trainer_stage2.predict(
        val_all_weapon_ds
    )
)


prob_weapon_all = (
    positive_probabilities(
        stage2_all_output.predictions
    )
)


true_1B = (
    val[
        "label_1B"
    ].to_numpy()
)


# ============================================================
# 22. END-TO-END HARD CASCADE THRESHOLD SEARCH
#
# DEVELOPMENT ONLY.
# ============================================================

best_stage2_threshold = 0.50
best_cascade_f1 = -1.0


for threshold in (
    THRESHOLDS
):

    pred_weapon = (
        prob_weapon_all
        >=
        threshold
    ).astype(int)


    cascade_pred = (

        (pred_1A == 1)

        &

        (pred_weapon == 1)

    ).astype(int)


    score = f1_score(

        true_1B,
        cascade_pred,

        zero_division=0
    )


    if score > best_cascade_f1:

        best_cascade_f1 = float(
            score
        )

        best_stage2_threshold = float(
            threshold
        )


# ============================================================
# 23. FINAL CASCADE
# ============================================================

pred_weapon_all = (
    prob_weapon_all
    >=
    best_stage2_threshold
).astype(int)


cascade_pred_1B = (

    (pred_1A == 1)

    &

    (pred_weapon_all == 1)

).astype(int)


cascade_precision = (
    precision_score(
        true_1B,
        cascade_pred_1B,
        zero_division=0
    )
)


cascade_recall = (
    recall_score(
        true_1B,
        cascade_pred_1B,
        zero_division=0
    )
)


cascade_f1 = (
    f1_score(
        true_1B,
        cascade_pred_1B,
        zero_division=0
    )
)


cm = confusion_matrix(

    true_1B,
    cascade_pred_1B,

    labels=[
        0,
        1
    ]
)


tn, fp, fn, tp = (
    cm.ravel()
)


# ============================================================
# 24. ORACLE-GATE DIAGNOSTIC
# ============================================================

oracle_pred_1B = (

    (true_1A == 1)

    &

    (pred_weapon_all == 1)

).astype(int)


oracle_precision = (
    precision_score(
        true_1B,
        oracle_pred_1B,
        zero_division=0
    )
)


oracle_recall = (
    recall_score(
        true_1B,
        oracle_pred_1B,
        zero_division=0
    )
)


oracle_f1 = (
    f1_score(
        true_1B,
        oracle_pred_1B,
        zero_division=0
    )
)


# ============================================================
# 25. ERROR PROPAGATION
# ============================================================

true_1B_mask = (
    true_1B == 1
)


lost_stage1 = int(

    np.sum(

        true_1B_mask

        &

        (pred_1A == 0)

    )
)


survived_stage1 = int(

    np.sum(

        true_1B_mask

        &

        (pred_1A == 1)

    )
)


# ============================================================
# 26. LOAD DIRECT BENCHMARK IF AVAILABLE
# ============================================================

direct_f1 = (
    DIRECT_F1_FALLBACK
)

direct_precision = np.nan
direct_recall = np.nan


if DIRECT_RESULTS_PATH.exists():

    try:

        direct_df = pd.read_csv(
            DIRECT_RESULTS_PATH
        )


        subset = direct_df[
            (
                direct_df[
                    "label"
                ].astype(str)
                ==
                "label_1B"
            )
            &
            (
                pd.to_numeric(
                    direct_df[
                        "seed"
                    ],
                    errors="coerce"
                )
                ==
                SEED
            )
        ].copy()


        if (
            "config"
            in subset.columns
        ):

            winner = subset[
                subset[
                    "config"
                ].astype(str)
                ==
                "weight_decay"
            ]

            if len(winner) > 0:

                subset = winner


        if (
            "learning_rate"
            in subset.columns
        ):

            lr_values = pd.to_numeric(
                subset[
                    "learning_rate"
                ],
                errors="coerce"
            )


            winner = subset[
                np.isclose(
                    lr_values,
                    3e-5,
                    atol=1e-10,
                    rtol=0
                )
            ]


            if len(winner) > 0:

                subset = winner


        if len(subset) > 0:

            best_index = (
                pd.to_numeric(
                    subset[
                        "tuned_f1"
                    ],
                    errors="coerce"
                )
                .idxmax()
            )


            row = subset.loc[
                best_index
            ]


            direct_f1 = float(
                row[
                    "tuned_f1"
                ]
            )


            if (
                "tuned_precision"
                in row.index
            ):

                direct_precision = float(
                    row[
                        "tuned_precision"
                    ]
                )


            if (
                "tuned_recall"
                in row.index
            ):

                direct_recall = float(
                    row[
                        "tuned_recall"
                    ]
                )


    except Exception as error:

        print(
            "\nCould not read direct "
            "benchmark CSV:"
        )

        print(error)


# ============================================================
# 27. FINAL RESULTS
# ============================================================

print(
    "\n\n" + "=" * 72
)

print(
    "UNWEIGHTED STAGE-2 — "
    "DEVELOPMENT RESULTS"
)

print(
    "=" * 72
)


print(
    "\nSTAGE 1 — GRIEVANCE"
)

print(
    f"Threshold: "
    f"{threshold_1A:.2f}"
)

print(
    f"Precision: "
    f"{stage1_precision:.3f}"
)

print(
    f"Recall: "
    f"{stage1_recall:.3f}"
)

print(
    f"F1: "
    f"{stage1_f1:.3f}"
)


print(
    "\nSTAGE 2 — GOLD GRIEVANCE SUBSET"
)

print(
    "Loss: UNWEIGHTED cross-entropy"
)

print(
    f"Threshold: "
    f"{conditional_threshold:.2f}"
)

print(
    f"Precision: "
    f"{conditional_precision:.3f}"
)

print(
    f"Recall: "
    f"{conditional_recall:.3f}"
)

print(
    f"F1: "
    f"{conditional_f1:.3f}"
)


print(
    "\nEND-TO-END HARD CASCADE"
)

print(
    f"Stage-2 threshold: "
    f"{best_stage2_threshold:.2f}"
)

print(
    f"Precision: "
    f"{cascade_precision:.3f}"
)

print(
    f"Recall: "
    f"{cascade_recall:.3f}"
)

print(
    f"F1: "
    f"{cascade_f1:.3f}"
)


print(
    "\nConfusion matrix:"
)

print(
    f"TN={tn} | FP={fp}"
)

print(
    f"FN={fn} | TP={tp}"
)


print(
    "\nERROR PROPAGATION"
)

print(
    f"True 1B total: "
    f"{int(true_1B.sum())}"
)

print(
    f"Survived Stage 1: "
    f"{survived_stage1}"
)

print(
    f"Lost at Stage 1: "
    f"{lost_stage1}"
)


print(
    "\nORACLE-GATE"
)

print(
    f"Precision: "
    f"{oracle_precision:.3f}"
)

print(
    f"Recall: "
    f"{oracle_recall:.3f}"
)

print(
    f"F1: "
    f"{oracle_f1:.3f}"
)


# ============================================================
# 28. MOST IMPORTANT COMPARISON
# ============================================================

print(
    "\n" + "-" * 72
)

print(
    "MODEL COMPARISON — SEED 42"
)

print(
    "-" * 72
)


print(
    "\nPREVIOUS WEIGHTED HIERARCHY"
)

print(
    "Conditional Stage-2:"
)

print(
    "P=.325 | R=1.000 | F1=.491"
)

print(
    "Hard cascade:"
)

print(
    "P=.233 | R=.925 | F1=.372"
)


print(
    "\nNEW UNWEIGHTED HIERARCHY"
)

print(
    f"Conditional Stage-2:"
)

print(
    f"P={conditional_precision:.3f} | "
    f"R={conditional_recall:.3f} | "
    f"F1={conditional_f1:.3f}"
)

print(
    "Hard cascade:"
)

print(
    f"P={cascade_precision:.3f} | "
    f"R={cascade_recall:.3f} | "
    f"F1={cascade_f1:.3f}"
)


print(
    "\nDIRECT DeBERTa"
)

if not np.isnan(
    direct_precision
):

    print(
        f"P={direct_precision:.3f}"
    )

if not np.isnan(
    direct_recall
):

    print(
        f"R={direct_recall:.3f}"
    )

print(
    f"F1={direct_f1:.3f}"
)


print(
    "\nChange vs weighted hierarchy:"
)

print(
    f"Conditional F1: "
    f"{conditional_f1 - 0.491:+.3f}"
)

print(
    f"Cascade F1: "
    f"{cascade_f1 - 0.372:+.3f}"
)


print(
    "\nChange vs direct model:"
)

print(
    f"Cascade F1: "
    f"{cascade_f1 - direct_f1:+.3f}"
)


# ============================================================
# 29. SIMPLE INTERPRETATION
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "INTERPRETATION"
)

print(
    "=" * 72
)


if (
    conditional_f1
    >
    0.491 + 0.03
):

    print(
        "\n✓ Removing Stage-2 class weights "
        "meaningfully improved the specialised task."
    )

else:

    print(
        "\n✗ Removing Stage-2 class weights "
        "did not meaningfully improve the "
        "specialised task."
    )


if (
    cascade_f1
    >
    direct_f1
):

    print(
        "✓ The unweighted hierarchy now "
        "beats the direct DeBERTa model."
    )

else:

    print(
        "✗ The unweighted hierarchy still "
        "does not beat direct DeBERTa."
    )


oracle_gap = (
    oracle_f1
    -
    cascade_f1
)


print(
    f"\nOracle-minus-cascade gap: "
    f"{oracle_gap:+.3f}"
)


if oracle_gap >= 0.05:

    print(
        "Stage-1 gating/error propagation "
        "remains important."
    )

else:

    print(
        "Stage 2 remains the more important "
        "limiting factor."
    )


# ============================================================
# 30. SAVE ROW-LEVEL RESULTS
# ============================================================

prediction_df = (
    val.copy()
)


prediction_df[
    "hier_unweighted_prob_1A"
] = prob_1A


prediction_df[
    "hier_unweighted_pred_1A"
] = pred_1A


prediction_df[
    "hier_unweighted_prob_weaponisation"
] = prob_weapon_all


prediction_df[
    "hier_unweighted_pred_weaponisation"
] = pred_weapon_all


prediction_df[
    "hier_unweighted_pred_1B"
] = cascade_pred_1B


prediction_df[
    "hier_unweighted_oracle_pred_1B"
] = oracle_pred_1B


prediction_df[
    "hier_unweighted_error"
] = np.select(

    [

        (
            (true_1B == 1)
            &
            (cascade_pred_1B == 1)
        ),

        (
            (true_1B == 0)
            &
            (cascade_pred_1B == 1)
        ),

        (
            (true_1B == 1)
            &
            (cascade_pred_1B == 0)
        )

    ],

    [
        "TP",
        "FP",
        "FN"
    ],

    default=
        "TN"
)


PREDICTION_PATH = (
    DATA_DIR
    /
    "results_deberta_hierarchical_1B_UNWEIGHTED_predictions_seed42.csv"
)


prediction_df.to_csv(
    PREDICTION_PATH,
    index=False
)


# ============================================================
# 31. SAVE SUMMARY
# ============================================================

summary = pd.DataFrame(
    [
        {

            "seed":
                SEED,

            "development_only":
                True,

            "stage2_loss":
                "unweighted_cross_entropy",

            "stage2_train_rows":
                n_stage2,

            "stage2_positive":
                n_positive,

            "stage2_negative":
                n_negative,

            "stage2_positive_rate":
                positive_rate,

            "stage1_threshold":
                threshold_1A,

            "stage1_precision":
                stage1_precision,

            "stage1_recall":
                stage1_recall,

            "stage1_f1":
                stage1_f1,

            "conditional_threshold":
                conditional_threshold,

            "conditional_precision":
                conditional_precision,

            "conditional_recall":
                conditional_recall,

            "conditional_f1":
                conditional_f1,

            "cascade_stage2_threshold":
                best_stage2_threshold,

            "cascade_precision":
                cascade_precision,

            "cascade_recall":
                cascade_recall,

            "cascade_f1":
                cascade_f1,

            "oracle_precision":
                oracle_precision,

            "oracle_recall":
                oracle_recall,

            "oracle_f1":
                oracle_f1,

            "tn":
                tn,

            "fp":
                fp,

            "fn":
                fn,

            "tp":
                tp,

            "true_1B_survived_stage1":
                survived_stage1,

            "true_1B_lost_stage1":
                lost_stage1,

            "previous_weighted_conditional_f1":
                0.491,

            "previous_weighted_cascade_f1":
                0.372,

            "direct_deberta_f1":
                direct_f1,

            "change_vs_weighted_conditional":
                conditional_f1
                -
                0.491,

            "change_vs_weighted_cascade":
                cascade_f1
                -
                0.372,

            "change_vs_direct":
                cascade_f1
                -
                direct_f1
        }
    ]
)


SUMMARY_PATH = (
    DATA_DIR
    /
    "results_deberta_hierarchical_1B_UNWEIGHTED_seed42.csv"
)


summary.to_csv(
    SUMMARY_PATH,
    index=False
)


print(
    "\nSaved:"
)

print(
    PREDICTION_PATH
)

print(
    SUMMARY_PATH
)


# ============================================================
# 32. CLEANUP
# ============================================================

del trainer_stage2
del stage2_model

clear_memory()


print(
    "\nExperiment complete."
)

## Tuning summary figures

The cells below reproduce the report's summary figures: the learning-rate search, the regularisation comparison, and the overall progression from baselines through the tuned transformers to the held-out test. Values are the threshold-tuned validation F1 scores recorded in the experiments above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lrs = [1, 2, 3]  # ×10⁻⁵
# threshold-tuned validation F1 from the learning-rate search (seed 42)
roberta = {"1A": [.618, .626, .655], "1B": [.469, .486, .538], "2": [.643, .667, .667]}
deberta = {"1A": [.644, .674, .657], "1B": [.211, .536, .545], "2": [.679, .735, .536]}
colours = {"1A": "#3f6fb0", "1B": "#c0504d", "2": "#4f8a5b"}

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, (name, data) in zip(axes, [("RoBERTa-base", roberta), ("DeBERTa-v3-base", deberta)]):
    for c, scores in data.items():
        ax.plot(lrs, scores, marker="o", label=c, color=colours[c])
        b = int(np.argmax(scores))                      # circle the best LR
        ax.scatter([lrs[b]], [scores[b]], s=130, facecolors="none",
                   edgecolors=colours[c], linewidths=1.8, zorder=5)
    ax.set_title(name, fontsize=11)
    ax.set_xticks(lrs, [f"{l}e-5" for l in lrs])
    ax.set_xlabel("Learning rate")
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, linewidth=0.5, alpha=0.4); ax.set_axisbelow(True)

axes[0].set_ylabel("Threshold-tuned validation F1")
axes[0].legend(title="Construct", frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig("rq1_lr_search.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

colours = {"1A": "#3f6fb0", "1B": "#c0504d", "2": "#4f8a5b"}

# --- learning-rate search (threshold-tuned validation F1, seed 42) ---
lrs = [1, 2, 3]  # ×10⁻⁵
lr_data = {
    "RoBERTa-base":    {"1A": [.618, .626, .655], "1B": [.469, .486, .538], "2": [.643, .667, .667]},
    "DeBERTa-v3-base": {"1A": [.644, .674, .657], "1B": [.211, .536, .545], "2": [.679, .735, .536]},
}
# --- regularisation search (at best LR) ---
configs = ["none", "weight\ndecay", "wd +\nwarm-up"]
reg_data = {
    "RoBERTa-base":    {"1A": [.655, .643, .635], "1B": [.538, .494, .565], "2": [.667, .676, .667]},
    "DeBERTa-v3-base": {"1A": [.674, .662, .662], "1B": [.545, .553, .464], "2": [.735, .732, .717]},
}

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharey=True)

# row 0: learning-rate lines
for j, (name, data) in enumerate(lr_data.items()):
    ax = axes[0, j]
    for c, s in data.items():
        ax.plot(lrs, s, marker="o", color=colours[c], label=c)
        b = int(np.argmax(s))
        ax.scatter([lrs[b]], [s[b]], s=120, facecolors="none",
                   edgecolors=colours[c], linewidths=1.8, zorder=5)
    ax.set_title(name, fontsize=11)
    ax.set_xticks(lrs, [f"{l}e-5" for l in lrs])
    ax.set_xlabel("Learning rate")

# row 1: regularisation bars
x = np.arange(len(configs)); w = 0.25
for j, (name, data) in enumerate(reg_data.items()):
    ax = axes[1, j]
    for i, (c, s) in enumerate(data.items()):
        ax.bar(x + (i - 1) * w, s, w, color=colours[c], edgecolor="white", linewidth=0.6, label=c)
    ax.set_xticks(x, configs)
    ax.set_xlabel("Regularisation")

for ax in axes.flat:
    ax.set_ylim(0, 0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, linewidth=0.5, alpha=0.4); ax.set_axisbelow(True)
axes[0, 0].set_ylabel("Validation F1\n(learning-rate search)")
axes[1, 0].set_ylabel("Validation F1\n(regularisation)")
axes[0, 0].legend(title="Construct", frameon=False, fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig("rq1_tuning_experiments.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

stages = ["Lexicon", "Naïve\nBayes", "Logistic\nregression",
          "Transformer\n(default)", "Transformer\n(tuned)", "Held-out\ntest"]
f1 = {
    "1A  grievance":   [.312, .535, .564, .625, .661, .612],
    "1B  weaponised":  [.169, .274, .382, .473, .540, .400],
    "2  conspiracy":   [.278, .416, .480, .678, .685, .716],
}
col = {"1A  grievance": "#3f6fb0", "1B  weaponised": "#c0504d", "2  conspiracy": "#4f8a5b"}

x = np.arange(len(stages))
fig, ax = plt.subplots(figsize=(9.6, 5.1))
ax.axvspan(-0.5, 2.5, color="#f1f1f1", zorder=0)   # baselines
ax.axvspan(2.5, 4.5, color="#eaf1f8", zorder=0)    # transformer development
ax.axvspan(4.5, 5.5, color="#e8f4ea", zorder=0)    # final test
for lab, s in f1.items():
    ax.plot(x, s, marker="o", color=col[lab], label=lab, lw=1.9, ms=6, zorder=3)
    ax.annotate(f"{s[-1]:.2f}", (x[-1], s[-1]), textcoords="offset points",
                xytext=(9, 0), fontsize=8.5, color=col[lab], va="center", fontweight="bold")
ax.set_xticks(x, stages, fontsize=8.7)
ax.set_xlim(-0.5, 5.9); ax.set_ylim(0, 0.82)
ax.set_ylabel("Positive-class F1")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5, alpha=0.4); ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=9, loc="lower right")
for xc, txt in [(1, "Baselines"),
                (3.5, "Transformers"),
                (5, "Test performance)")]:
    ax.text(xc, 0.795, txt, ha="center", fontsize=8, style="italic", color="#555")
plt.tight_layout()
plt.savefig("rq1_experiment_progression.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C  = {"1A": "#3f6fb0", "1B": "#c0504d", "2": "#4f8a5b"}
CN = {"1A": "1A grievance", "1B": "1B weaponised", "2": "2 conspiracy"}
arch = ["RoBERTa-base", "DeBERTa-v3-base"]

# --- Row A: learning-rate search (tuned validation F1, seed 42) ---
lrs = [1, 2, 3]  # ×10⁻⁵
lrF1 = {"RoBERTa-base":    {"1A": [.618,.626,.655], "1B": [.469,.486,.538], "2": [.643,.667,.667]},
        "DeBERTa-v3-base": {"1A": [.644,.674,.657], "1B": [.211,.536,.545], "2": [.679,.735,.536]}}
lrBest = {"RoBERTa-base": {"1A":2,"1B":2,"2":1}, "DeBERTa-v3-base": {"1A":1,"1B":2,"2":1}}  # index of chosen LR

# --- Row B: regularisation search (tuned F1, at best LR) ---
cfgs = ["none", "weight\ndecay", "wd +\nwarm-up"]
regF1 = {"RoBERTa-base":    {"1A": [.655,.643,.635], "1B": [.538,.494,.565], "2": [.667,.676,.667]},
         "DeBERTa-v3-base": {"1A": [.674,.662,.662], "1B": [.545,.553,.464], "2": [.735,.732,.717]}}

# --- Row C: 3-seed confirmation (mean ± SD, winning config) ---
seedM = {"RoBERTa-base": {"1A":.661,"1B":.524,"2":.655}, "DeBERTa-v3-base": {"1A":.659,"1B":.540,"2":.685}}
seedS = {"RoBERTa-base": {"1A":.009,"1B":.030,"2":.019}, "DeBERTa-v3-base": {"1A":.014,"1B":.021,"2":.038}}
selected = {"RoBERTa-base": ["1A"], "DeBERTa-v3-base": ["1B", "2"]}   # ★ final architecture per construct

fig, axes = plt.subplots(3, 2, figsize=(9.5, 10), sharey=True)
for j, a in enumerate(arch):
    # A: learning-rate lines
    ax = axes[0, j]
    for c in ["1A", "1B", "2"]:
        ax.plot(lrs, lrF1[a][c], marker="o", color=C[c], lw=1.7, ms=5, label=CN[c])
        bi = lrBest[a][c]
        ax.scatter([lrs[bi]], [lrF1[a][c][bi]], s=110, facecolors="none", edgecolors=C[c], lw=1.8, zorder=5)
    ax.set_xticks(lrs, [f"{l}e-5" for l in lrs]); ax.set_title(a, fontsize=12, fontweight="bold", pad=10)
    if j == 0: ax.set_ylabel("A  Learning rate\n\nValidation F1", fontsize=9.5)
    # B: regularisation bars
    ax = axes[1, j]; x = np.arange(3); w = 0.26
    for k, c in enumerate(["1A", "1B", "2"]):
        ax.bar(x + (k-1)*w, regF1[a][c], w, color=C[c], edgecolor="white", lw=.5)
    ax.set_xticks(x, cfgs, fontsize=8)
    if j == 0: ax.set_ylabel("B  Regularisation\n\nValidation F1", fontsize=9.5)
    # C: seed confirmation points
    ax = axes[2, j]
    for k, c in enumerate(["1A", "1B", "2"]):
        ax.errorbar([k], [seedM[a][c]], yerr=[seedS[a][c]], fmt="o", ms=9, color=C[c], capsize=5, lw=1.6)
        if c in selected[a]:
            ax.text(k, seedM[a][c] + seedS[a][c] + 0.045, "★", ha="center", fontsize=15, color="#1f3864")
    ax.set_xticks(np.arange(3), ["1A", "1B", "2"]); ax.set_xlim(-0.5, 2.5)
    if j == 0: ax.set_ylabel("C  Seed confirmation\n(mean ± SD, 3 seeds)\n\nValidation F1", fontsize=9.5)

for ax in axes.flat:
    ax.set_ylim(0.15, 0.82); ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, lw=0.5, alpha=0.35); ax.set_axisbelow(True)
axes[0, 1].legend(frameon=False, fontsize=8, loc="lower right")
fig.text(0.5, 0.005, "○ selected learning rate      ★ selected architecture", ha="center", fontsize=8.5, color="#555")
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig("rq1_transformer_experiments.png", dpi=300, bbox_inches="tight")
plt.show()